# Southwest Airlines analytics

Execution guide added 2026-11-16. All original code cells and saved outputs are preserved. These outputs are historical evidence, not a new run.

**Do not Run all yet.** S01 depends on saved checkpoints; S03 and S09 require input; S15 pins an older BTS dataset; S26 targets a historical test database. Read the execution guide below.

# Execution guide

## Prerequisites
Use Colab with Google Drive access, the project folder and preserved BTS input. SEC acquisition prompts for a contact email. Keep credentials out of shared outputs. PostgreSQL setup installs packages and creates a local database role. The database lives in the temporary Colab runtime; Drive backups persist.

## Cell numbering
The labels S01–S27 added above code cells identify original saved code positions. Inline comments such as CELL 3 predate the recovery cell and do not equal notebook positions. Documentation cells do not change these labels.

## Fresh acquisition route
Skip S01, which requires existing checkpoints. Run S02 setup, S03 SEC acquisition, S04 financial validation, S05 expense breakdown, S06 annual comparison, S07 filing text extraction, S08 reviewed findings, S09 BTS upload, S10 BTS validation, S11 coverage/KPIs, S12 cause analysis, S13 airport analysis. Stop at any failed validation.

S08 contains reviewed findings tied to existing passages; do not label new passages reviewed automatically. S09 requires an official BTS ZIP or CSV upload, not a generated dataset. All new runs must retain source provenance.

## Existing-checkpoint route
Run S01 to restore the pinned validated BTS checkpoint and constants. It does not restore every financial variable or PostgreSQL. Run S13 for the checkpoint-based airport analysis, or S14–S17 for SQL setup/loading/reporting. S12 needs the dependent validated state described in its code; S01 alone is not a universal recovery cell.

## SQL and reporting
S14 installs/starts PostgreSQL and configures credentials. S15 loads the pinned BTS CSV and creates a backup. S16 runs annual SQL validation. S17 creates the monthly view and dashboard export. S18–S21 are optional download/mount/inspection helpers.

Important: S15 explicitly pins BTS run 20261113T180223779285Z. Fresh acquisition can create a newer run, but SQL loading will still use this historical checkpoint until configured. Do not claim that a fresh BTS upload automatically flows through every downstream step.

## Forecast evaluation
S22 inspects the monthly export. S23 validates it, freezes forecasts using 2023, and evaluates 2024. It uses a pinned monthly export. Outputs are timestamped. Keep test-year evaluation separate from future tuning and do not call these forecasts official budgets.

## Recovery testing
S24 checks the backup and installed PostgreSQL. S25 restores the pinned backup into a newly named isolated test database and compares annual totals. S26 recreates the monthly view and compares dashboard values.

S26 currently hardcodes southwest_restore_20261115t184531406040z and its report directory. Before a NEW recovery test, set TEST_DB to the test_db created by S25 and REPORT_DIR to S25's report_dir. Otherwise it targets the previous test database. The historical test passed; automatic chaining still needs implementation.

S27 lists saved project artifacts. Do not use Run all blindly: recovery comes first physically, uploads require input, and several later cells pin prior run IDs.

## Desktop deliverables
Power BI uses exported CSVs, not a live connection to Colab's localhost. Preserve table names MonthlyOperations, Financials2024 and ExpenseComparison when refreshing. Financial cards cover 2024; expense comparisons cover both years. Excel needs the local CSV path configured in Power Query and an .xlsm file saved to a local filesystem path for the supplied VBA export.

## Verification and restart
After a Colab reset, remount Drive and reinstall/start PostgreSQL as needed. Preserve existing archives. Use a separate database for restore tests. Do not delete backups or rewrite raw data to get checks to pass. Save validation manifests alongside each run. A reproducible release still needs a clean-runtime test with captured dependency versions and explicit input configuration.


## S01 — Checkpoint recovery — existing saved data only

Original code position 1; existing inline CELL labels may differ.


In [ ]:
# RECOVERY CELL — Run after a Colab runtime restart.
# Restores the saved BTS data needed for Cell 11.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/southwest_analytics")

# Restore financial constants for any later financial cells.
ACCESSION = "0000092380-25-000024"
PERIOD_START = "2024-01-01"
PERIOD_END = "2024-12-31"

FILING_URL = (
    "https://investors.southwest.com/sec-filings/all-sec-filings/"
    "content/0000092380-25-000024/luv-20241231.htm"
)

# Restore the exact validated BTS checkpoint from your successful run.
bts_run_id = "20261113T180223779285Z"

clean_path = (
    PROJECT / f"data/processed/bts_arrivals_2023_2024_{bts_run_id}.csv"
)
checks_path = (
    PROJECT / f"reports/validation/bts_checks_{bts_run_id}.csv"
)

for path in [clean_path, checks_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Saved checkpoint not found: {path}. "
            "Do not recreate or substitute the data."
        )

saved_checks = pd.read_csv(checks_path)

if saved_checks.empty or not saved_checks["status"].eq("PASS").all():
    raise ValueError("The saved BTS validation report is not fully passing.")

bts = pd.read_csv(
    clean_path,
    dtype={
        "carrier": "string",
        "airport": "string",
        "source_snapshot": "string"
    },
    parse_dates=["period"]
)

snapshots = bts["source_snapshot"].dropna().unique()

if len(snapshots) != 1:
    raise ValueError("Expected exactly one BTS source snapshot.")

bts_raw_path = PROJECT / "data/raw/bts" / str(snapshots[0])

if not bts_raw_path.exists():
    raise FileNotFoundError(f"Original BTS source not found: {bts_raw_path}")

# Restore logging.
logger = logging.getLogger("southwest_analytics")
logger.setLevel(logging.INFO)
logger.propagate = False

for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

handler = logging.FileHandler(
    PROJECT / "logs/source_audit.log",
    encoding="utf-8"
)
handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
logger.addHandler(handler)

logger.info("Restored BTS checkpoint: %s", clean_path.name)

print("BTS checkpoint loaded.")
print(f"Rows: {len(bts):,}")
print(f"Years: {sorted(bts['year'].unique().tolist())}")
print(f"Airports: {bts['airport'].nunique()}")
print("Saved validation report: all checks PASS")

## S02 — Project setup

Original code position 2; existing inline CELL labels may differ.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import logging

import pandas as pd
import requests
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/southwest_analytics")

for folder in [
    "data/raw/sec",
    "data/processed",
    "reports/validation",
    "logs",
    "docs"
]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("southwest_analytics")
logger.setLevel(logging.INFO)
logger.propagate = False

# Avoid duplicate log handlers when rerunning the cell.
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

handler = logging.FileHandler(
    PROJECT / "logs/source_audit.log", encoding="utf-8"
)
handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
logger.addHandler(handler)

print("Project folder:", PROJECT)

## S03 — SEC companyfacts acquisition — interactive contact input

Original code position 3; existing inline CELL labels may differ.


In [ ]:
from getpass import getpass

SEC_URL = (
    "https://data.sec.gov/api/xbrl/companyfacts/"
    "CIK0000092380.json"
)

contact_email = getpass("Your contact email for the SEC request: ").strip()

if "@" not in contact_email or "." not in contact_email.split("@")[-1]:
    raise ValueError("Enter a valid contact email.")

session = requests.Session()
session.headers.update({
    "User-Agent": f"SouthwestAnalytics academic-project {contact_email}",
    "Accept": "application/json"
})

try:
    response = session.get(SEC_URL, timeout=60)
    response.raise_for_status()
    companyfacts = response.json()

    if int(companyfacts.get("cik", -1)) != 92380:
        raise ValueError("Unexpected company identifier in SEC response.")

    if "us-gaap" not in companyfacts.get("facts", {}):
        raise ValueError("SEC response does not contain expected financial facts.")

except (requests.RequestException, ValueError):
    logger.exception("SEC collection failed")
    raise

finally:
    session.close()

retrieved_at = datetime.now(timezone.utc)
run_id = retrieved_at.strftime("%Y%m%dT%H%M%S%fZ")
raw_path = PROJECT / "data/raw/sec" / f"companyfacts_{run_id}.json"

raw_path.write_bytes(response.content)

manifest = {
    "source": "SEC EDGAR Company Facts",
    "source_url": SEC_URL,
    "company": companyfacts["entityName"],
    "cik": companyfacts["cik"],
    "retrieved_at_utc": retrieved_at.isoformat(),
    "raw_file": str(raw_path.relative_to(PROJECT)),
    "sha256": hashlib.sha256(response.content).hexdigest(),
    "http_status": response.status_code,
    "bytes": len(response.content)
}

manifest_path = raw_path.with_suffix(".manifest.json")
manifest_path.write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

logger.info("SEC source preserved: %s", raw_path.name)


print("Company:", companyfacts["entityName"])
print("Raw source saved:", raw_path.name)
print("SHA-256:", manifest["sha256"])

## S04 — 2024 financial extraction and reconciliation

Original code position 4; existing inline CELL labels may differ.


In [ ]:

# CELL 3 — Extract and validate Southwest's 2024 financial figures
# Requires Cell 1 and Cell 2 to have run successfully.

ACCESSION = "0000092380-25-000024"
PERIOD_START = "2024-01-01"
PERIOD_END = "2024-12-31"

FILING_URL = (
    "https://investors.southwest.com/sec-filings/all-sec-filings/"
    "content/0000092380-25-000024/luv-20241231.htm"
)

# Benchmarks transcribed from the audited 2024 income statement.
# These values validate extracted data; they never fill missing data.
METRICS = {
    "operating_revenue": {
        "tags": [
            "RevenueFromContractWithCustomerExcludingAssessedTax",
            "Revenues",
            "SalesRevenueNet"
        ],
        "expected_usd": 27_483_000_000
    },
    "operating_expenses": {
        "tags": ["OperatingCostsAndExpenses"],
        "expected_usd": 27_162_000_000
    },
    "operating_income": {
        "tags": ["OperatingIncomeLoss"],
        "expected_usd": 321_000_000
    },
    "net_income": {
        "tags": ["NetIncomeLoss"],
        "expected_usd": 465_000_000
    }
}

gaap = companyfacts["facts"]["us-gaap"]

extracted = []
checks = []
validated_rows = []

for metric, config in METRICS.items():
    candidates = []

    for tag in config["tags"]:
        records = gaap.get(tag, {}).get("units", {}).get("USD", [])

        for record in records:
            if (
                record.get("accn") == ACCESSION
                and record.get("form") == "10-K"
                and record.get("start") == PERIOD_START
                and record.get("end") == PERIOD_END
            ):
                candidates.append({
                    "metric": metric,
                    "namespace": "us-gaap",
                    "tag": tag,
                    "value_usd": record["val"],
                    "period_start": record["start"],
                    "period_end": record["end"],
                    "accession": record["accn"],
                    "filed": record.get("filed"),
                    "unit": "USD",
                    "source_snapshot": raw_path.name
                })

    # Preserve candidate evidence, even when validation fails.
    extracted.extend(candidates)
    unique_values = {row["value_usd"] for row in candidates}

    if not candidates:
        actual = None
        status = "FAIL_MISSING"

    elif len(unique_values) != 1:
        actual = None
        status = "FAIL_CONFLICT"

    else:
        actual = next(iter(unique_values))
        status = (
            "PASS"
            if actual == config["expected_usd"]
            else "FAIL_RECONCILIATION"
        )

    checks.append({
        "metric": metric,
        "actual_usd": actual,
        "expected_usd": config["expected_usd"],
        "status": status,
        "benchmark_source": FILING_URL
    })

    if status == "PASS":
        validated_rows.append({
            "metric": metric,
            "value_usd": actual,
            "unit": "USD",
            "period_start": PERIOD_START,
            "period_end": PERIOD_END,
            "accession": ACCESSION,
            "source_tags": "|".join(
                sorted({row["tag"] for row in candidates})
            ),
            "source_snapshot": raw_path.name
        })

# Save extraction evidence and validation results.
candidate_df = pd.DataFrame(
    extracted,
    columns=[
        "metric", "namespace", "tag", "value_usd",
        "period_start", "period_end", "accession",
        "filed", "unit", "source_snapshot"
    ]
)

validation_df = pd.DataFrame(checks)

# Nullable integers preserve dollar values without scientific notation.
validation_df["actual_usd"] = pd.array(
    validation_df["actual_usd"], dtype="Int64"
)
validation_df["expected_usd"] = pd.array(
    validation_df["expected_usd"], dtype="Int64"
)

candidate_path = (
    PROJECT
    / f"reports/validation/financial_candidates_{run_id}.csv"
)
validation_path = (
    PROJECT
    / f"reports/validation/financial_checks_{run_id}.csv"
)
financial_path = (
    PROJECT
    / f"data/processed/financials_2024_{run_id}.csv"
)

candidate_df.to_csv(candidate_path, index=False)
validation_df.to_csv(validation_path, index=False)

print("FINANCIAL SOURCE RECONCILIATION")

display(
    validation_df[
        ["metric", "actual_usd", "expected_usd", "status"]
    ].style.format(
        {
            "actual_usd": "{:,.0f}",
            "expected_usd": "{:,.0f}"
        },
        na_rep="MISSING"
    )
)

# Stop before producing an approved dataset if any check fails.
if not validation_df["status"].eq("PASS").all():
    # Remove an earlier output for this same run if it exists.
    financial_path.unlink(missing_ok=True)

    failed_metrics = validation_df.loc[
        validation_df["status"].ne("PASS"), "metric"
    ].tolist()

    logger.error(
        "Financial source audit failed for: %s", failed_metrics
    )

    raise ValueError(
        f"Audit failed for {failed_metrics}. "
        "Inspect the saved candidate and validation reports."
    )

financials = pd.DataFrame(validated_rows)
financials["value_usd"] = pd.array(
    financials["value_usd"], dtype="Int64"
)

values = financials.set_index("metric")["value_usd"]

# Independent accounting consistency check using extracted values.
calculated_operating_income = int(
    values["operating_revenue"] - values["operating_expenses"]
)
reported_operating_income = int(values["operating_income"])

identity_passed = (
    calculated_operating_income == reported_operating_income
)

identity_report = pd.DataFrame([{
    "check": "Revenue minus operating expenses equals operating income",
    "calculated_usd": calculated_operating_income,
    "reported_usd": reported_operating_income,
    "difference_usd": (
        calculated_operating_income - reported_operating_income
    ),
    "status": "PASS" if identity_passed else "FAIL"
}])

identity_report.to_csv(
    PROJECT / f"reports/validation/accounting_identity_{run_id}.csv",
    index=False
)

if not identity_passed:
    financial_path.unlink(missing_ok=True)
    logger.error("Operating income accounting identity failed.")

    raise ValueError(
        "Revenue minus operating expenses does not equal "
        "reported operating income."
    )

# Publish the processed data only after all checks pass.
financials.to_csv(financial_path, index=False)

operating_margin = (
    int(values["operating_income"])
    / int(values["operating_revenue"])
    * 100
)

logger.info(
    "2024 financial audit passed. Operating margin: %.4f%%",
    operating_margin
)

print("\nSource reconciliation: PASSED")
print("Revenue − operating expenses = operating income: PASSED")
print(f"2024 operating margin: {operating_margin:.2f}%")
print("\nValidated financial data saved to:")
print(financial_path)

## S05 — 2024 expense breakdown

Original code position 5; existing inline CELL labels may differ.


In [ ]:
# CELL 4 — Analyze and reconcile 2024 operating expenses
# Run after Cell 3 passes.

import matplotlib.pyplot as plt

EXPENSE_TAGS = {
    "Salaries, wages and benefits": "LaborAndRelatedExpense",
    "Fuel and oil": "FuelCosts",
    "Maintenance": "AircraftMaintenanceMaterialsAndRepairs",
    "Landing fees and rentals": "LandingFeesAndOtherRentals",
    "Depreciation and amortization": "DepreciationAndAmortization",
    "Other operating expenses": "OtherCostAndExpenseOperating"
}

expense_rows = []
extraction_checks = []

for category, tag in EXPENSE_TAGS.items():
    records = gaap.get(tag, {}).get("units", {}).get("USD", [])

    matches = [
        record
        for record in records
        if (
            record.get("accn") == ACCESSION
            and record.get("form") == "10-K"
            and record.get("start") == PERIOD_START
            and record.get("end") == PERIOD_END
        )
    ]

    unique_values = {record["val"] for record in matches}

    if not matches:
        status = "FAIL_MISSING"
    elif len(unique_values) != 1:
        status = "FAIL_CONFLICT"
    else:
        status = "PASS"

    extraction_checks.append({
        "category": category,
        "tag": tag,
        "status": status
    })

    if status == "PASS":
        expense_rows.append({
            "category": category,
            "tag": tag,
            "expense_usd": int(next(iter(unique_values))),
            "unit": "USD",
            "period_start": PERIOD_START,
            "period_end": PERIOD_END,
            "accession": ACCESSION,
            "source_snapshot": raw_path.name
        })

extraction_report = pd.DataFrame(extraction_checks)

output_path = (
    PROJECT / f"data/processed/expense_breakdown_2024_{run_id}.csv"
)

extraction_report.to_csv(
    PROJECT / f"reports/validation/expense_extraction_{run_id}.csv",
    index=False
)

if not extraction_report["status"].eq("PASS").all():
    output_path.unlink(missing_ok=True)
    display(extraction_report)
    logger.error("Expense category extraction failed.")
    raise ValueError(
        "Missing or conflicting expense facts. Review the extraction report."
    )

expenses = pd.DataFrame(expense_rows)

# Use the validated reported total from Cell 3.
reported_total = int(
    financials.set_index("metric").loc[
        "operating_expenses", "value_usd"
    ]
)

category_total = int(expenses["expense_usd"].sum())
difference = category_total - reported_total

reconciliation = pd.DataFrame([{
    "check": "Expense categories sum to reported operating expenses",
    "category_total_usd": category_total,
    "reported_total_usd": reported_total,
    "difference_usd": difference,
    "status": "PASS" if difference == 0 else "FAIL"
}])

reconciliation.to_csv(
    PROJECT / f"reports/validation/expense_total_check_{run_id}.csv",
    index=False
)

if difference != 0:
    output_path.unlink(missing_ok=True)
    logger.error("Expense breakdown reconciliation failed.")
    display(reconciliation)
    raise ValueError(
        "Expense categories do not match the reported total. "
        "Do not publish the breakdown."
    )

expenses["share_of_operating_expenses_pct"] = (
    expenses["expense_usd"] / reported_total * 100
)

expenses = expenses.sort_values(
    "expense_usd", ascending=False
).reset_index(drop=True)

expenses.to_csv(output_path, index=False)

print("EXPENSE BREAKDOWN — SOUTHWEST, 2024")
display(
    expenses[
        ["category", "expense_usd", "share_of_operating_expenses_pct"]
    ].style.format({
        "expense_usd": "${:,.0f}",
        "share_of_operating_expenses_pct": "{:.2f}%"
    })
)

print(f"\nSum of categories: USD {category_total:,.0f}")
print(f"Reported total:    USD {reported_total:,.0f}")
print(f"Difference:        USD {difference:,.0f}")
print("Expense reconciliation: PASSED")

# Plot only the reconciled data.
plot_data = expenses.sort_values("expense_usd")

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    plot_data["category"],
    plot_data["expense_usd"] / 1_000_000_000,
    color="#2563EB"
)

ax.bar_label(bars, fmt="%.2f", padding=4)
ax.set_title("Southwest Airlines — 2024 Operating Expense Breakdown")
ax.set_xlabel("USD billions")
ax.set_ylabel("")
ax.set_xlim(
    0,
    (plot_data["expense_usd"].max() / 1_000_000_000) * 1.15
)
ax.spines[["top", "right"]].set_visible(False)

fig.text(
    0.01, 0.01,
    "Source: SEC Company Facts; 2024 Form 10-K, "
    "accession 0000092380-25-000024.",
    fontsize=8,
    color="#555555"
)

fig.tight_layout(rect=[0, 0.05, 1, 1])

chart_path = (
    PROJECT / f"reports/expense_breakdown_2024_{run_id}.png"
)
fig.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

logger.info("2024 expense breakdown reconciled and saved.")

print("\nSaved data:", output_path)
print("Saved chart:", chart_path)

## S06 — 2023–2024 expense comparison

Original code position 6; existing inline CELL labels may differ.


In [ ]:
# CELL 5 — Compare 2023 and 2024 operating expenses
# Requires successful Cells 1–4.
# Uses comparative annual figures from the SAME 2024 Form 10-K.

comparison_rows = []
extraction_checks = []

tags_to_extract = {
    **EXPENSE_TAGS,
    "Total operating expenses": "OperatingCostsAndExpenses"
}

for year in [2023, 2024]:
    period_start = f"{year}-01-01"
    period_end = f"{year}-12-31"

    for category, tag in tags_to_extract.items():
        records = gaap.get(tag, {}).get("units", {}).get("USD", [])

        matches = [
            record
            for record in records
            if (
                record.get("accn") == ACCESSION
                and record.get("form") == "10-K"
                and record.get("start") == period_start
                and record.get("end") == period_end
            )
        ]

        unique_values = {record["val"] for record in matches}

        if not matches:
            status = "FAIL_MISSING"
        elif len(unique_values) != 1:
            status = "FAIL_CONFLICT"
        else:
            status = "PASS"

        extraction_checks.append({
            "year": year,
            "category": category,
            "tag": tag,
            "status": status
        })

        if status == "PASS":
            comparison_rows.append({
                "year": year,
                "category": category,
                "tag": tag,
                "expense_usd": int(next(iter(unique_values))),
                "unit": "USD",
                "period_start": period_start,
                "period_end": period_end,
                "accession": ACCESSION,
                "source_snapshot": raw_path.name
            })

extraction_report = pd.DataFrame(extraction_checks)

comparison_path = (
    PROJECT / f"data/processed/expense_comparison_2023_2024_{run_id}.csv"
)
history_path = (
    PROJECT / f"data/processed/expense_history_2023_2024_{run_id}.csv"
)

# Remove earlier outputs for this run until validation succeeds.
comparison_path.unlink(missing_ok=True)
history_path.unlink(missing_ok=True)

extraction_report.to_csv(
    PROJECT / f"reports/validation/expense_comparison_extraction_{run_id}.csv",
    index=False
)

if not extraction_report["status"].eq("PASS").all():
    display(extraction_report)
    logger.error("Expense comparison extraction failed.")
    raise ValueError(
        "Missing or conflicting facts. Review the extraction report."
    )

expense_history = pd.DataFrame(comparison_rows)

# Reconcile each year's categories against its reported total.
annual_checks = []

for year in [2023, 2024]:
    year_data = expense_history[expense_history["year"].eq(year)]

    category_sum = int(
        year_data.loc[
            year_data["category"].ne("Total operating expenses"),
            "expense_usd"
        ].sum()
    )

    reported_total = int(
        year_data.loc[
            year_data["category"].eq("Total operating expenses"),
            "expense_usd"
        ].iloc[0]
    )

    annual_checks.append({
        "year": year,
        "category_sum_usd": category_sum,
        "reported_total_usd": reported_total,
        "difference_usd": category_sum - reported_total,
        "status": "PASS" if category_sum == reported_total else "FAIL"
    })

annual_reconciliation = pd.DataFrame(annual_checks)

annual_reconciliation.to_csv(
    PROJECT / f"reports/validation/expense_comparison_totals_{run_id}.csv",
    index=False
)

display(annual_reconciliation)

if not annual_reconciliation["status"].eq("PASS").all():
    logger.error("Annual expense reconciliation failed.")
    raise ValueError("Expense categories do not reconcile for both years.")

# One row per category, with separate columns for each year.
comparison = (
    expense_history[
        expense_history["category"].ne("Total operating expenses")
    ]
    .pivot(index="category", columns="year", values="expense_usd")
    .rename(columns={
        2023: "expense_2023_usd",
        2024: "expense_2024_usd"
    })
    .reset_index()
)

comparison.columns.name = None

comparison["change_usd"] = (
    comparison["expense_2024_usd"]
    - comparison["expense_2023_usd"]
)

# A zero prior-year expense has no defined percentage growth.
denominator = comparison["expense_2023_usd"].astype(float)
denominator = denominator.where(denominator.ne(0))

comparison["change_pct"] = (
    comparison["change_usd"] / denominator * 100
)

reported_totals = (
    expense_history[
        expense_history["category"].eq("Total operating expenses")
    ]
    .set_index("year")["expense_usd"]
)

total_change = int(reported_totals.loc[2024] - reported_totals.loc[2023])
category_change = int(comparison["change_usd"].sum())

bridge_passed = category_change == total_change

pd.DataFrame([{
    "check": "Category changes explain total expense change",
    "category_change_usd": category_change,
    "reported_total_change_usd": total_change,
    "difference_usd": category_change - total_change,
    "status": "PASS" if bridge_passed else "FAIL"
}]).to_csv(
    PROJECT / f"reports/validation/expense_change_bridge_{run_id}.csv",
    index=False
)

if not bridge_passed:
    logger.error("Expense change bridge failed.")
    raise ValueError("Category changes do not explain the total change.")

comparison = comparison.sort_values(
    "change_usd", ascending=False
).reset_index(drop=True)

# Save only after all validation checks pass.
expense_history.to_csv(history_path, index=False)
comparison.to_csv(comparison_path, index=False)

print("\nOPERATING EXPENSE CHANGES: 2023 → 2024")
display(
    comparison.style.format({
        "expense_2023_usd": "${:,.0f}",
        "expense_2024_usd": "${:,.0f}",
        "change_usd": "${:+,.0f}",
        "change_pct": "{:+.2f}%"
    }, na_rep="N/A")
)

print(f"\nTotal operating expense change: USD {total_change:+,}")
print("Category changes reconcile to total change: PASSED")

# Positive = expense increase; negative = expense decrease.
plot_data = comparison.sort_values("change_usd")
colors = [
    "#DC6B45" if value > 0 else "#27857A"
    for value in plot_data["change_usd"]
]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    plot_data["category"],
    plot_data["change_usd"] / 1_000_000,
    color=colors
)

ax.bar_label(bars, fmt="%+.0f", padding=5)
ax.axvline(0, color="#555555", linewidth=0.8)
ax.margins(x=0.20)
ax.set_title("Southwest — Change in Operating Expenses, 2023 to 2024")
ax.set_xlabel("Change in USD millions")
ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)

fig.text(
    0.01, 0.01,
    "Source: SEC Company Facts; comparative figures in the 2024 Form 10-K.",
    fontsize=8,
    color="#555555"
)

fig.tight_layout(rect=[0, 0.05, 1, 1])

chart_path = (
    PROJECT / f"reports/expense_changes_2023_2024_{run_id}.png"
)
fig.savefig(chart_path, dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

logger.info("2023–2024 expense comparison validated and saved.")

print("\nSaved comparison:", comparison_path)
print("Saved chart:", chart_path)

## S07 — Filing text extraction

Original code position 7; existing inline CELL labels may differ.


In [ ]:

%pip -q install beautifulsoup4

from bs4 import BeautifulSoup
import re

filing_directory = PROJECT / "data/raw/filings"
filing_directory.mkdir(parents=True, exist_ok=True)

filing_path = filing_directory / f"southwest_2024_10k_{ACCESSION}.html"

if filing_path.exists():
    html_bytes = filing_path.read_bytes()
    print("Using the saved annual report.")

else:
    try:
        response = requests.get(
            FILING_URL,
            headers={
                "User-Agent": "SouthwestAnalytics academic research",
                "Accept": "text/html"
            },
            timeout=60
        )
        response.raise_for_status()
        html_bytes = response.content

        preview = BeautifulSoup(html_bytes, "html.parser").get_text(
            " ", strip=True
        )
        preview = re.sub(r"\s+", " ", preview)

        if (
            "Southwest" not in preview
            or "2024 Compared with 2023" not in preview
        ):
            raise ValueError(
                "The response does not contain the expected report section."
            )

        filing_path.write_bytes(html_bytes)

        source_record = {
            "source_url": FILING_URL,
            "accession": ACCESSION,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "sha256": hashlib.sha256(html_bytes).hexdigest(),
            "raw_file": str(filing_path.relative_to(PROJECT))
        }

        filing_path.with_suffix(".manifest.json").write_text(
            json.dumps(source_record, indent=2),
            encoding="utf-8"
        )

        logger.info("Annual report downloaded and preserved.")

    except (requests.RequestException, ValueError):
        logger.exception("Annual report collection failed.")
        raise

soup = BeautifulSoup(html_bytes, "html.parser")

for element in soup(["script", "style"]):
    element.decompose()

paragraphs = []
seen = set()

for element in soup.find_all(["p", "div"]):
    if element.find_parent("table") is not None:
        continue

    if element.find(["p", "div", "table"]) is not None:
        continue

    text = re.sub(
        r"\s+", " ", element.get_text(" ", strip=True)
    ).strip()

    if text and text not in seen:
        paragraphs.append(text)
        seen.add(text)

# Locate the annual comparison section.
starts = [
    i for i, text in enumerate(paragraphs)
    if text.casefold() == "2024 compared with 2023"
]

if len(starts) != 1:
    raise ValueError(
        f"Expected one 2024 comparison heading; found {len(starts)}. "
        "Stop here so we can inspect the document structure."
    )

start = starts[0]

# Limit the search to this section. Stop at the next major section.
end_markers = {
    "2023 compared with 2022",
    "liquidity and capital resources"
}

end = next(
    (
        i for i in range(start + 1, len(paragraphs))
        if paragraphs[i].casefold() in end_markers
    ),
    None
)

if end is None:
    raise ValueError(
        "Could not identify the end of the comparison section. "
        "Stop here rather than mixing sections."
    )

patterns = {
    "Labor": r"\bsalaries\b|\bwages\b|\blabor\b|\bprofitsharing\b",
    "Fuel": r"\bfuel\b",
    "Maintenance": r"\bmaintenance\b"
}

evidence_rows = []

for position in range(start + 1, end):
    paragraph = paragraphs[position]

    # Skip headings and short labels.
    if len(paragraph) < 180:
        continue

    topics = [
        topic for topic, pattern in patterns.items()
        if re.search(pattern, paragraph, flags=re.IGNORECASE)
    ]

    if not topics:
        continue

    evidence_rows.append({
        "passage_id": f"MD&A-{position:04d}",
        "topics": "; ".join(topics),
        "section": "2024 Compared with 2023",
        "source_text": paragraph,
        "source_url": FILING_URL,
        "accession": ACCESSION,
        "source_snapshot": filing_path.name,
        "review_status": "PENDING",
        "analyst_interpretation": ""
    })

if not evidence_rows:
    raise ValueError(
        "No candidate passages found. Inspect the extraction before continuing."
    )

commentary_evidence = pd.DataFrame(evidence_rows)

evidence_path = (
    PROJECT / f"reports/management_commentary_candidates_{run_id}.csv"
)
commentary_evidence.to_csv(evidence_path, index=False)

print(f"\nFound {len(commentary_evidence)} candidate passages.")

for row in commentary_evidence.itertuples(index=False):
    print(f"\n[{row.passage_id}] {row.topics}")
    print(row.source_text)
    print("-" * 70)

logger.info(
    "Saved %s candidate management commentary passages.",
    len(commentary_evidence)
)

print("\nEvidence saved:", evidence_path)
print("Review status: PENDING — extraction is not interpretation.")

## S08 — Reviewed management findings

Original code position 8; existing inline CELL labels may differ.


In [ ]:
# CELL 7 — Save reviewed financial findings and supporting evidence
# Requires successful Cells 5 and 6.
#
# These are analyst-reviewed summaries of management commentary,
# not independent proof of causation.

review_specs = [
    {
        "category": "Salaries, wages and benefits",
        "passage_ids": ["MD&A-0678"],
        "explanation": (
            "Management attributes the increase mainly to step/pay-rate "
            "increases, including ratified labor contracts, with additional "
            "increases from health care and other benefits."
        ),
        "review_note": (
            "The passage explains the direction of the change but does "
            "not provide a complete dollar allocation by driver."
        ),
        "required_phrases": ["step/pay rate increases"]
    },
    {
        "category": "Fuel and oil",
        "passage_ids": ["MD&A-0683", "MD&A-0685"],
        "explanation": (
            "Management attributes the decrease primarily to lower fuel "
            "prices, partially offset by increased gallons consumed."
        ),
        "review_note": (
            "Two passages form one paragraph across a page break. "
            "Economic fuel cost and hedging disclosures must not be "
            "treated as interchangeable with GAAP fuel expense."
        ),
        "required_phrases": [
            "increase in fuel gallons consumed",
            "activities compared"
        ]
    },
    {
        "category": "Maintenance",
        "passage_ids": ["MD&A-0696"],
        "explanation": (
            "Management identifies increased engine shop visits and "
            "other engine expenses, airframe repair costs and timing, "
            "and seat-refurbishment materials as major drivers."
        ),
        "review_note": (
            "The approximate driver percentages in the passage do not "
            "allocate the entire increase. No residual cause is assumed."
        ),
        "required_phrases": ["engine shop visits"]
    }
]

if not commentary_evidence["passage_id"].is_unique:
    raise ValueError("Duplicate passage IDs found. Review the evidence table.")

if not comparison["category"].is_unique:
    raise ValueError("Duplicate expense categories found in the comparison.")

evidence_lookup = commentary_evidence.set_index("passage_id")
financial_lookup = comparison.set_index("category")

reviewed_evidence = commentary_evidence.copy()
reviewed_evidence["review_note"] = ""
reviewed_evidence["review_method"] = ""

review_time = datetime.now(timezone.utc).isoformat()
finding_rows = []

for spec in review_specs:
    category = spec["category"]
    passage_ids = spec["passage_ids"]

    missing_ids = [
        passage_id
        for passage_id in passage_ids
        if passage_id not in evidence_lookup.index
    ]

    if missing_ids:
        raise ValueError(
            f"Missing source passages for {category}: {missing_ids}"
        )

    if category not in financial_lookup.index:
        raise ValueError(f"Missing financial comparison for {category}")

    passages = evidence_lookup.loc[passage_ids]

    if not passages["accession"].eq(ACCESSION).all():
        raise ValueError(f"Unexpected filing accession for {category}")

    # Preserve the exact extracted source text.
    combined_text = "\n\n".join(passages["source_text"].tolist())

    # Guard against passage IDs pointing to different content after a rerun.
    normalized_text = " ".join(combined_text.casefold().split())

    for phrase in spec["required_phrases"]:
        if phrase.casefold() not in normalized_text:
            raise ValueError(
                f"Source wording changed for {category}. "
                "Review the passages before saving."
            )

    financial_row = financial_lookup.loc[category]

    finding_rows.append({
        "category": category,
        "expense_2023_usd": int(financial_row["expense_2023_usd"]),
        "expense_2024_usd": int(financial_row["expense_2024_usd"]),
        "change_usd": int(financial_row["change_usd"]),
        "change_pct": float(financial_row["change_pct"]),
        "management_explanation": spec["explanation"],
        "evidence_passage_ids": "|".join(passage_ids),
        "source_text": combined_text,
        "review_note": spec["review_note"],
        "attribution": "Management explanation",
        "causality_independently_verified": False,
        "review_method": "Assistant review of user-supplied extraction",
        "reviewed_at_utc": review_time,
        "accession": ACCESSION,
        "source_url": FILING_URL,
        "financial_source_snapshot": raw_path.name
    })

    selected = reviewed_evidence["passage_id"].isin(passage_ids)

    reviewed_evidence.loc[
        selected, "review_status"
    ] = "REVIEWED_SUPPORTING"

    reviewed_evidence.loc[
        selected, "analyst_interpretation"
    ] = spec["explanation"]

    reviewed_evidence.loc[
        selected, "review_note"
    ] = spec["review_note"]

    reviewed_evidence.loc[
        selected, "review_method"
    ] = "Assistant review of user-supplied extraction"

# Correct the keyword-filter false positive without deleting the passage.
false_positive_id = "MD&A-0701"

if false_positive_id in evidence_lookup.index:
    selected = reviewed_evidence["passage_id"].eq(false_positive_id)

    reviewed_evidence.loc[
        selected, "review_status"
    ] = "EXCLUDED_FROM_AIRCRAFT_MAINTENANCE"

    reviewed_evidence.loc[selected, "review_note"] = (
        "This passage concerns other operating expenses, including "
        "technology maintenance agreements. It does not explain the "
        "separate aircraft maintenance expense category."
    )

    reviewed_evidence.loc[selected, "review_method"] = (
        "Assistant review of user-supplied extraction"
    )

reviewed_findings = pd.DataFrame(finding_rows)

findings_path = (
    PROJECT / f"reports/reviewed_expense_findings_{run_id}.csv"
)
reviewed_evidence_path = (
    PROJECT / f"reports/reviewed_commentary_evidence_{run_id}.csv"
)

reviewed_findings.to_csv(findings_path, index=False)
reviewed_evidence.to_csv(reviewed_evidence_path, index=False)

print("REVIEWED FINANCIAL FINDINGS")

display(
    reviewed_findings[[
        "category",
        "change_usd",
        "change_pct",
        "management_explanation",
        "evidence_passage_ids"
    ]].style.format({
        "change_usd": "${:+,.0f}",
        "change_pct": "{:+.2f}%"
    })
)

print("\nEVIDENCE REVIEW STATUS")
display(
    reviewed_evidence["review_status"]
    .value_counts()
    .rename_axis("review_status")
    .reset_index(name="passages")
)

logger.info(
    "Saved %s reviewed expense findings with source references.",
    len(reviewed_findings)
)

print("\nSaved findings:", findings_path)
print("Saved reviewed evidence:", reviewed_evidence_path)
print("\nUnreviewed passages remain PENDING.")

## S09 — BTS source upload — manual input

Original code position 9; existing inline CELL labels may differ.


In [ ]:
# CELL 8 — Preserve and inspect the official BTS source download
# Requires the project setup from Cell 1.

from google.colab import files
from io import BytesIO
import zipfile

BTS_SOURCE_URL = (
    "https://www.transtats.bts.gov/OT_Delay/OT_DelayCause1.asp"
)

print("Upload the BTS ZIP or CSV downloaded for Southwest, 2023–2024.")
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload exactly one BTS source file.")

original_name, source_bytes = next(iter(uploaded.items()))
extension = Path(original_name).suffix.lower()

if extension not in {".zip", ".csv"}:
    raise ValueError("Expected a ZIP or CSV file from BTS.")

bts_run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
bts_raw_directory = PROJECT / "data/raw/bts"
bts_raw_directory.mkdir(parents=True, exist_ok=True)

bts_raw_path = (
    bts_raw_directory / f"arrival_statistics_{bts_run_id}{extension}"
)
bts_raw_path.write_bytes(source_bytes)

# Read the CSV inside the ZIP without extracting archive paths.
if extension == ".zip":
    with zipfile.ZipFile(BytesIO(source_bytes)) as archive:
        csv_members = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
            and not name.startswith("__MACOSX/")
        ]

        if len(csv_members) != 1:
            raise ValueError(
                f"Expected one CSV inside the ZIP; found {csv_members}. "
                "The original file is saved. Inspect the archive before proceeding."
            )

        csv_member = csv_members[0]
        csv_bytes = archive.read(csv_member)
else:
    csv_member = original_name
    csv_bytes = source_bytes

# Preserve codes as text during the initial source audit.
# Numeric and date conversions come after schema inspection.
bts_raw = pd.read_csv(
    BytesIO(csv_bytes),
    dtype="string",
    encoding="utf-8-sig"
)

bts_manifest = {
    "source": "BTS Airline On-Time Statistics and Delay Causes",
    "source_url": BTS_SOURCE_URL,
    "collection_method": "User download from official source",
    "uploaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "original_filename": original_name,
    "raw_file": str(bts_raw_path.relative_to(PROJECT)),
    "csv_member": csv_member,
    "sha256": hashlib.sha256(source_bytes).hexdigest(),
    "requested_carrier": "WN",
    "requested_start_month": "2023-01",
    "requested_end_month": "2024-12",
    "requested_airports": "All",
    "filter_validation_status": "PENDING",
    "rows": len(bts_raw),
    "columns": bts_raw.columns.tolist()
}

bts_raw_path.with_suffix(".manifest.json").write_text(
    json.dumps(bts_manifest, indent=2),
    encoding="utf-8"
)

bts_profile = pd.DataFrame({
    "column": bts_raw.columns,
    "missing_rows": bts_raw.isna().sum().to_numpy(),
    "distinct_non_null_values": bts_raw.nunique().to_numpy()
})

bts_profile.to_csv(
    PROJECT / f"reports/validation/bts_source_profile_{bts_run_id}.csv",
    index=False
)

print(f"\nRows: {len(bts_raw):,}")
print(f"Columns: {len(bts_raw.columns)}")
print(f"Exact duplicate rows: {bts_raw.duplicated().sum():,}")

print("\nCOLUMN NAMES")
print(bts_raw.columns.tolist())

print("\nFIRST FIVE ROWS")
display(bts_raw.head())

print("\nSOURCE PROFILE")
display(bts_profile)

logger.info("BTS source preserved and profiled: %s", bts_raw_path.name)

print("\nOriginal source saved:", bts_raw_path)
print("Carrier and date coverage still require validation.")

## S10 — BTS validation

Original code position 10; existing inline CELL labels may differ.


In [ ]:
# CELL 9 — Validate BTS carrier, dates, keys and numerical consistency
# Requires Cell 8.

import numpy as np

bts = bts_raw.copy()

count_columns = [
    "arr_flights", "arr_del15", "arr_cancelled", "arr_diverted"
]

cause_count_columns = [
    "carrier_ct", "weather_ct", "nas_ct",
    "security_ct", "late_aircraft_ct"
]

delay_columns = [
    "arr_delay", "carrier_delay", "weather_delay",
    "nas_delay", "security_delay", "late_aircraft_delay"
]

numeric_columns = (
    ["year", "month"]
    + count_columns
    + cause_count_columns
    + delay_columns
)

required_columns = (
    ["carrier", "carrier_name", "airport", "airport_name"]
    + numeric_columns
)

missing_columns = sorted(set(required_columns) - set(bts.columns))

if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

# Standardize identifiers without changing the preserved raw source.
for column in ["carrier", "airport"]:
    bts[column] = bts[column].str.strip().str.upper()

# Invalid numeric strings become NaN and fail validation below.
for column in numeric_columns:
    bts[column] = pd.to_numeric(bts[column], errors="coerce")

checks = []

def add_check(name, passed, detail=""):
    checks.append({
        "check": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "detail": detail
    })

add_check(
    "Required fields are populated",
    bts[required_columns].notna().all().all()
)

add_check(
    "Numeric values are finite",
    np.isfinite(
        bts[numeric_columns].to_numpy(dtype=float)
    ).all()
)

add_check(
    "Only Southwest carrier WN is present",
    bts["carrier"].eq("WN").all()
)

add_check(
    "Airport codes are three uppercase letters",
    bts["airport"].str.fullmatch(r"[A-Z]{3}", na=False).all()
)

add_check(
    "Years are limited to 2023 and 2024",
    bts["year"].isin([2023, 2024]).all()
)

add_check(
    "Months are integers from 1 to 12",
    (
        bts["month"].between(1, 12)
        & bts["month"].mod(1).eq(0)
    ).all()
)

add_check(
    "Flight counts are non-negative integers",
    (
        bts[count_columns].ge(0)
        & bts[count_columns].mod(1).eq(0)
    ).all().all()
)

add_check(
    "Delay minutes and attributed cause counts are non-negative",
    bts[cause_count_columns + delay_columns].ge(0).all().all()
)

# Cause counts are fractional allocations, so do NOT round them to integers.
key_columns = ["year", "month", "carrier", "airport"]
duplicate_keys = bts.duplicated(key_columns, keep=False)

add_check(
    "Carrier-airport-month keys are unique",
    not duplicate_keys.any(),
    f"Rows with duplicate keys: {int(duplicate_keys.sum())}"
)

observed_periods = set(
    bts[["year", "month"]].dropna()
    .itertuples(index=False, name=None)
)

expected_periods = {
    (year, month)
    for year in [2023, 2024]
    for month in range(1, 13)
}

add_check(
    "All 24 requested months are present",
    observed_periods == expected_periods,
    f"Missing months: {sorted(expected_periods - observed_periods)}"
)

# These categories should not exceed the reported flight population.
classified_flights = bts[
    ["arr_del15", "arr_cancelled", "arr_diverted"]
].sum(axis=1, min_count=3)

add_check(
    "Delayed, cancelled and diverted counts fit within total flights",
    classified_flights.le(bts["arr_flights"]).all()
)

# Five cause counts are published to two decimal places.
# Allow up to 0.03 flights for cumulative rounding differences.
cause_count_difference = (
    bts[cause_count_columns].sum(axis=1, min_count=5)
    - bts["arr_del15"]
)

add_check(
    "Attributed delay counts reconcile within rounding tolerance",
    cause_count_difference.abs().le(0.030001).all(),
    f"Maximum difference: {cause_count_difference.abs().max():.6f}"
)

cause_minutes = [
    "carrier_delay", "weather_delay", "nas_delay",
    "security_delay", "late_aircraft_delay"
]

minute_difference = (
    bts[cause_minutes].sum(axis=1, min_count=5)
    - bts["arr_delay"]
)

add_check(
    "Cause delay minutes sum to total delay minutes",
    minute_difference.abs().le(0.000001).all(),
    f"Maximum difference: {minute_difference.abs().max():.6f}"
)

validation = pd.DataFrame(checks)

validation_path = (
    PROJECT / f"reports/validation/bts_checks_{bts_run_id}.csv"
)
clean_path = (
    PROJECT / f"data/processed/bts_arrivals_2023_2024_{bts_run_id}.csv"
)

validation.to_csv(validation_path, index=False)

print("BTS VALIDATION RESULTS")
display(validation)

# Save problematic rows for investigation, not silent removal.
issue_mask = (
    duplicate_keys
    | cause_count_difference.abs().gt(0.030001)
    | minute_difference.abs().gt(0.000001)
    | classified_flights.gt(bts["arr_flights"])
    | bts[numeric_columns].isna().any(axis=1)
)

bts.loc[issue_mask].to_csv(
    PROJECT / f"reports/validation/bts_exception_rows_{bts_run_id}.csv",
    index=False
)

if not validation["status"].eq("PASS").all():
    clean_path.unlink(missing_ok=True)
    logger.error("BTS validation failed.")
    raise ValueError(
        "BTS validation failed. Share the failed checks before continuing."
    )

# Convert validated integer fields.
for column in ["year", "month"] + count_columns:
    bts[column] = bts[column].astype("int64")

bts["period"] = pd.to_datetime({
    "year": bts["year"],
    "month": bts["month"],
    "day": 1
})

# Audit airport-month coverage without filling absent combinations.
all_periods = pd.date_range("2023-01-01", "2024-12-01", freq="MS")

expected_grid = pd.MultiIndex.from_product(
    [sorted(bts["airport"].unique()), all_periods],
    names=["airport", "period"]
)

observed_grid = pd.MultiIndex.from_frame(
    bts[["airport", "period"]]
)

absent_combinations = (
    expected_grid.difference(observed_grid).to_frame(index=False)
)

absent_combinations["review_status"] = "PENDING"
absent_combinations["note"] = (
    "Absent source row; service absence versus missing reporting "
    "has not been established."
)

absent_combinations.to_csv(
    PROJECT / f"reports/validation/bts_absent_airport_months_{bts_run_id}.csv",
    index=False
)

bts = bts.sort_values(["period", "airport"]).reset_index(drop=True)
bts["source_snapshot"] = bts_raw_path.name
bts.to_csv(clean_path, index=False)

# Update the source record with verified coverage.
bts_manifest["filter_validation_status"] = "PASS"
bts_manifest["validated_at_utc"] = datetime.now(timezone.utc).isoformat()
bts_manifest["observed_carriers"] = sorted(bts["carrier"].unique().tolist())
bts_manifest["observed_months"] = len(observed_periods)
bts_manifest["observed_airports"] = int(bts["airport"].nunique())

bts_raw_path.with_suffix(".manifest.json").write_text(
    json.dumps(bts_manifest, indent=2),
    encoding="utf-8"
)

print("\nABSENT AIRPORT–MONTH COMBINATIONS")
display(absent_combinations)

print(f"\nValidated rows: {len(bts):,}")
print(f"Months covered: {len(observed_periods)}")
print(f"Airports represented: {bts['airport'].nunique()}")
print("Saved validated data:", clean_path)

logger.info("BTS structural validation passed: %s rows.", len(bts))

## S11 — Coverage review and operational KPIs

Original code position 11; existing inline CELL labels may differ.


In [ ]:
# CELL 10 — Review coverage exceptions and calculate operational KPIs
# Requires successful Cell 9.

if not validation["status"].eq("PASS").all():
    raise ValueError("Complete BTS validation before calculating KPIs.")

# -------------------------------------------------------------
# 1. Document the service exits explaining absent airport-months.
# -------------------------------------------------------------

exit_announcement_url = (
    "https://investors.southwest.com/news-events/press-releases/"
    "detail/72/southwest-airlines-reports-first-quarter-2024-results"
)

exit_confirmation_url = FILING_URL

coverage_review = absent_combinations.copy()
coverage_review["period"] = pd.to_datetime(coverage_review["period"])

explained = (
    coverage_review["airport"].isin(["BLI", "IAH", "SYR"])
    & coverage_review["period"].between("2024-09-01", "2024-12-01")
)

coverage_review.loc[
    explained, "review_status"
] = "EXPLAINED_SERVICE_EXIT"

coverage_review.loc[explained, "note"] = (
    "Absence is consistent with Southwest's announced August 4, 2024 "
    "service exit and the 2024 annual report's confirmation of the exit. "
    "No zero-flight records have been inserted."
)

coverage_review.loc[
    explained, "announcement_source"
] = exit_announcement_url

coverage_review.loc[
    explained, "confirmation_source"
] = exit_confirmation_url

coverage_review.to_csv(
    PROJECT / f"reports/validation/bts_coverage_review_{bts_run_id}.csv",
    index=False
)

print("COVERAGE REVIEW")
display(
    coverage_review[
        ["airport", "period", "review_status"]
    ]
)

# -------------------------------------------------------------
# 2. Aggregate counts FIRST, then calculate rates.
# -------------------------------------------------------------

sum_columns = [
    "arr_flights",
    "arr_del15",
    "arr_cancelled",
    "arr_diverted",
    "arr_delay"
]

monthly_kpis = (
    bts.groupby("period", as_index=False)[sum_columns]
    .sum()
    .sort_values("period")
)

annual_kpis = (
    bts.groupby("year", as_index=False)[sum_columns]
    .sum()
    .sort_values("year")
)

def calculate_kpis(table):
    result = table.copy()

    result["on_time_flights"] = (
        result["arr_flights"]
        - result["arr_del15"]
        - result["arr_cancelled"]
        - result["arr_diverted"]
    )

    flight_denominator = result["arr_flights"].where(
        result["arr_flights"].gt(0)
    )
    delayed_denominator = result["arr_del15"].where(
        result["arr_del15"].gt(0)
    )

    result["cancellation_rate_pct"] = (
        result["arr_cancelled"] / flight_denominator * 100
    )
    result["delayed_flight_share_pct"] = (
        result["arr_del15"] / flight_denominator * 100
    )
    result["diversion_rate_pct"] = (
        result["arr_diverted"] / flight_denominator * 100
    )
    result["on_time_flight_share_pct"] = (
        result["on_time_flights"] / flight_denominator * 100
    )
    result["avg_delay_minutes_per_delayed_arrival"] = (
        result["arr_delay"] / delayed_denominator
    )

    result["scope"] = "BTS reported domestic arrivals; carrier WN"
    result["source_snapshot"] = bts_raw_path.name

    return result

monthly_kpis = calculate_kpis(monthly_kpis)
annual_kpis = calculate_kpis(annual_kpis)

# -------------------------------------------------------------
# 3. Verify aggregation preserves the source totals.
# -------------------------------------------------------------

aggregation_checks = []

for column in sum_columns:
    raw_total = float(bts[column].sum())
    monthly_total = float(monthly_kpis[column].sum())
    annual_total = float(annual_kpis[column].sum())

    passed = (
        np.isclose(raw_total, monthly_total, rtol=0, atol=0.001)
        and np.isclose(raw_total, annual_total, rtol=0, atol=0.001)
    )

    aggregation_checks.append({
        "metric": column,
        "source_total": raw_total,
        "monthly_total": monthly_total,
        "annual_total": annual_total,
        "status": "PASS" if passed else "FAIL"
    })

aggregation_report = pd.DataFrame(aggregation_checks)

aggregation_report.to_csv(
    PROJECT / f"reports/validation/bts_kpi_aggregation_{bts_run_id}.csv",
    index=False
)

if not aggregation_report["status"].eq("PASS").all():
    raise ValueError("Aggregated totals do not reconcile to the source.")

# -------------------------------------------------------------
# 4. Compare annual rates in percentage points.
# -------------------------------------------------------------

annual_index = annual_kpis.set_index("year")

rate_columns = [
    "cancellation_rate_pct",
    "delayed_flight_share_pct",
    "diversion_rate_pct",
    "on_time_flight_share_pct"
]

rate_changes = pd.DataFrame([
    {
        "metric": column,
        "2023_pct": annual_index.loc[2023, column],
        "2024_pct": annual_index.loc[2024, column],
        "change_percentage_points": (
            annual_index.loc[2024, column]
            - annual_index.loc[2023, column]
        )
    }
    for column in rate_columns
])

# -------------------------------------------------------------
# 5. Save and display.
# -------------------------------------------------------------

monthly_kpis.to_csv(
    PROJECT / f"data/processed/bts_monthly_kpis_{bts_run_id}.csv",
    index=False
)
annual_kpis.to_csv(
    PROJECT / f"data/processed/bts_annual_kpis_{bts_run_id}.csv",
    index=False
)
rate_changes.to_csv(
    PROJECT / f"reports/bts_annual_rate_changes_{bts_run_id}.csv",
    index=False
)

print("\nANNUAL OPERATIONAL PERFORMANCE")
display(
    annual_kpis[[
        "year",
        "arr_flights",
        "arr_cancelled",
        "arr_del15",
        "cancellation_rate_pct",
        "delayed_flight_share_pct",
        "on_time_flight_share_pct",
        "avg_delay_minutes_per_delayed_arrival"
    ]].style.format({
        "arr_flights": "{:,.0f}",
        "arr_cancelled": "{:,.0f}",
        "arr_del15": "{:,.0f}",
        "cancellation_rate_pct": "{:.2f}%",
        "delayed_flight_share_pct": "{:.2f}%",
        "on_time_flight_share_pct": "{:.2f}%",
        "avg_delay_minutes_per_delayed_arrival": "{:.2f}"
    })
)

print("\nYEAR-OVER-YEAR RATE CHANGES")
display(
    rate_changes.style.format({
        "2023_pct": "{:.2f}%",
        "2024_pct": "{:.2f}%",
        "change_percentage_points": "{:+.2f}"
    })
)

logger.info("Monthly and annual BTS KPIs calculated and reconciled.")

print("\nAggregation reconciliation: PASSED")
print("Monthly KPIs, annual KPIs and rate changes saved.")
print("Scope: reported domestic arrivals, not company-wide operations.")

## S12 — Delay causes and duration changes

Original code position 12; existing inline CELL labels may differ.


In [ ]:
# CELL 11 — Explain the change in average delay duration
# Requires successful Cells 9 and 10.

CAUSES = {
    "Carrier": "carrier_delay",
    "Weather": "weather_delay",
    "National aviation system": "nas_delay",
    "Security": "security_delay",
    "Late-arriving aircraft": "late_aircraft_delay"
}

required = ["year", "arr_del15", "arr_delay"] + list(CAUSES.values())

missing = sorted(set(required) - set(bts.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Aggregate the original validated airport-month data.
annual_delays = (
    bts.groupby("year")[required[1:]]
    .sum()
    .sort_index()
)

if set(annual_delays.index) != {2023, 2024}:
    raise ValueError("Expected both 2023 and 2024.")

if annual_delays["arr_del15"].le(0).any():
    raise ValueError("Cannot calculate averages without delayed arrivals.")

# Reconcile attributed cause minutes to total delay minutes.
cause_totals = annual_delays[list(CAUSES.values())].sum(axis=1)

checks = pd.DataFrame({
    "year": annual_delays.index,
    "cause_minutes": cause_totals.to_numpy(),
    "reported_delay_minutes": annual_delays["arr_delay"].to_numpy()
})

checks["difference_minutes"] = (
    checks["cause_minutes"] - checks["reported_delay_minutes"]
)
checks["status"] = np.where(
    checks["difference_minutes"].abs().le(0.000001),
    "PASS",
    "FAIL"
)

checks.to_csv(
    PROJECT / f"reports/validation/delay_cause_checks_{bts_run_id}.csv",
    index=False
)

if not checks["status"].eq("PASS").all():
    display(checks)
    raise ValueError("Delay-cause minutes do not reconcile.")

# Build one record per cause and year.
rows = []

for year in [2023, 2024]:
    delayed_arrivals = int(annual_delays.loc[year, "arr_del15"])
    total_delay_minutes = float(annual_delays.loc[year, "arr_delay"])

    for cause, column in CAUSES.items():
        cause_minutes = float(annual_delays.loc[year, column])

        rows.append({
            "year": year,
            "cause": cause,
            "delay_minutes": cause_minutes,
            "delayed_arrivals_all_causes": delayed_arrivals,
            "share_of_delay_minutes_pct": (
                cause_minutes / total_delay_minutes * 100
                if total_delay_minutes > 0 else np.nan
            ),
            "minutes_per_delayed_arrival": (
                cause_minutes / delayed_arrivals
            ),
            "source_snapshot": bts_raw_path.name,
            "scope": "BTS reported domestic arrivals; carrier WN"
        })

cause_analysis = pd.DataFrame(rows)

duration_comparison = (
    cause_analysis.pivot(
        index="cause",
        columns="year",
        values="minutes_per_delayed_arrival"
    )
    .rename(columns={
        2023: "minutes_2023",
        2024: "minutes_2024"
    })
    .reset_index()
)
duration_comparison.columns.name = None

duration_comparison["change_minutes"] = (
    duration_comparison["minutes_2024"]
    - duration_comparison["minutes_2023"]
)

duration_comparison = duration_comparison.sort_values(
    "change_minutes", ascending=False
).reset_index(drop=True)

# Confirm the contributions explain the full average-duration change.
overall_average = (
    annual_delays["arr_delay"] / annual_delays["arr_del15"]
)

overall_change = float(
    overall_average.loc[2024] - overall_average.loc[2023]
)
explained_change = float(duration_comparison["change_minutes"].sum())

change_check = pd.DataFrame([{
    "overall_average_change_minutes": overall_change,
    "sum_of_cause_changes_minutes": explained_change,
    "difference_minutes": explained_change - overall_change,
    "status": (
        "PASS"
        if np.isclose(
            explained_change, overall_change, rtol=0, atol=0.000001
        )
        else "FAIL"
    )
}])

change_check.to_csv(
    PROJECT / f"reports/validation/delay_duration_change_check_{bts_run_id}.csv",
    index=False
)

if change_check.loc[0, "status"] != "PASS":
    raise ValueError("Cause contributions do not explain the average change.")

cause_analysis.to_csv(
    PROJECT / f"data/processed/bts_delay_causes_{bts_run_id}.csv",
    index=False
)

duration_comparison.to_csv(
    PROJECT / f"reports/delay_duration_comparison_{bts_run_id}.csv",
    index=False
)

print("DELAY MINUTES BY CAUSE AND YEAR")
display(
    cause_analysis[[
        "year", "cause", "delay_minutes",
        "share_of_delay_minutes_pct",
        "minutes_per_delayed_arrival"
    ]].style.format({
        "delay_minutes": "{:,.0f}",
        "share_of_delay_minutes_pct": "{:.2f}%",
        "minutes_per_delayed_arrival": "{:.2f}"
    })
)

print("\nCONTRIBUTIONS TO THE CHANGE IN AVERAGE DELAY")
display(
    duration_comparison.style.format({
        "minutes_2023": "{:.2f}",
        "minutes_2024": "{:.2f}",
        "change_minutes": "{:+.2f}"
    })
)

print(f"\n2023 average delay: {overall_average.loc[2023]:.2f} minutes")
print(f"2024 average delay: {overall_average.loc[2024]:.2f} minutes")
print(f"Change: {overall_change:+.2f} minutes")
print("Cause-minute reconciliation: PASSED")
print("Average-duration change reconciliation: PASSED")

# Visualize each category's contribution to the change.
plot_data = duration_comparison.sort_values("change_minutes")

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    plot_data["cause"],
    plot_data["change_minutes"],
    color=[
        "#DC6B45" if value > 0 else "#27857A"
        for value in plot_data["change_minutes"]
    ]
)

ax.bar_label(bars, fmt="%+.2f", padding=5)
ax.axvline(0, color="#555555", linewidth=0.8)
ax.margins(x=0.25)
ax.set_title("Southwest: Contributions to Average Delay Change, 2023–2024")
ax.set_xlabel("Change in attributed minutes per delayed arrival")
ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)

fig.text(
    0.01, 0.01,
    "Source: BTS domestic arrivals, WN. "
    "All categories use the same delayed-arrival denominator within each year.",
    fontsize=8
)

fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(
    PROJECT / f"reports/delay_duration_changes_{bts_run_id}.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

logger.info("Delay-cause analysis completed and reconciled.")

## S13 — Airport and monthly delay concentration

Original code position 13; existing inline CELL labels may differ.


In [ ]:
# CELL 12 — Locate late-aircraft delay concentration
# Requires the recovery cell or validated BTS data from Cell 9.

required_columns = [
    "year", "month", "period", "airport", "airport_name",
    "arr_flights", "arr_del15", "late_aircraft_delay"
]

missing = sorted(set(required_columns) - set(bts.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if bts.duplicated(["year", "month", "carrier", "airport"]).any():
    raise ValueError("Duplicate carrier-airport-month keys found.")

# -------------------------------------------------------------
# 1. Aggregate airport-year counts before calculating measures.
# -------------------------------------------------------------

airport_year = (
    bts.groupby(["airport", "year"], as_index=False)
    .agg(
        months_present=("month", "nunique"),
        reported_flights=("arr_flights", "sum"),
        delayed_arrivals=("arr_del15", "sum"),
        late_aircraft_minutes=("late_aircraft_delay", "sum")
    )
)

# Use the most recent observed name for display only.
airport_names = (
    bts.sort_values("period")
    .drop_duplicates("airport", keep="last")
    [["airport", "airport_name"]]
)

airport_year = airport_year.merge(
    airport_names,
    on="airport",
    how="left",
    validate="many_to_one"
)

airport_year["late_minutes_per_100_flights"] = (
    airport_year["late_aircraft_minutes"]
    / airport_year["reported_flights"].where(
        airport_year["reported_flights"].gt(0)
    )
    * 100
)

airport_year["late_minutes_per_delayed_arrival"] = (
    airport_year["late_aircraft_minutes"]
    / airport_year["delayed_arrivals"].where(
        airport_year["delayed_arrivals"].gt(0)
    )
)

year_totals = airport_year.groupby("year")[
    "late_aircraft_minutes"
].transform("sum")

airport_year["share_of_year_late_minutes_pct"] = (
    airport_year["late_aircraft_minutes"]
    / year_totals.where(year_totals.gt(0))
    * 100
)

# -------------------------------------------------------------
# 2. Aggregate monthly network totals.
# -------------------------------------------------------------

monthly_late = (
    bts.groupby("period", as_index=False)
    .agg(
        reported_flights=("arr_flights", "sum"),
        delayed_arrivals=("arr_del15", "sum"),
        late_aircraft_minutes=("late_aircraft_delay", "sum")
    )
    .sort_values("period")
)

monthly_late["late_minutes_per_100_flights"] = (
    monthly_late["late_aircraft_minutes"]
    / monthly_late["reported_flights"].where(
        monthly_late["reported_flights"].gt(0)
    )
    * 100
)

# -------------------------------------------------------------
# 3. Reconcile airport and monthly totals against source data.
# -------------------------------------------------------------

aggregation_checks = []

for source_column, output_column in [
    ("arr_flights", "reported_flights"),
    ("arr_del15", "delayed_arrivals"),
    ("late_aircraft_delay", "late_aircraft_minutes")
]:
    for year in [2023, 2024]:
        source_total = float(
            bts.loc[bts["year"].eq(year), source_column].sum()
        )
        airport_total = float(
            airport_year.loc[
                airport_year["year"].eq(year), output_column
            ].sum()
        )
        monthly_total = float(
            monthly_late.loc[
                monthly_late["period"].dt.year.eq(year), output_column
            ].sum()
        )

        passed = (
            np.isclose(source_total, airport_total, rtol=0, atol=0.001)
            and np.isclose(
                source_total, monthly_total, rtol=0, atol=0.001
            )
        )

        aggregation_checks.append({
            "year": year,
            "metric": source_column,
            "source_total": source_total,
            "airport_total": airport_total,
            "monthly_total": monthly_total,
            "status": "PASS" if passed else "FAIL"
        })

aggregation_report = pd.DataFrame(aggregation_checks)

aggregation_report.to_csv(
    PROJECT / f"reports/validation/late_aircraft_aggregation_{bts_run_id}.csv",
    index=False
)

if not aggregation_report["status"].eq("PASS").all():
    display(aggregation_report)
    raise ValueError("Airport or monthly totals failed reconciliation.")

# -------------------------------------------------------------
# 4. Identify airports present in all 24 months.
# -------------------------------------------------------------

coverage = airport_year.pivot(
    index="airport",
    columns="year",
    values="months_present"
).reindex(columns=[2023, 2024])

comparable_airports = coverage.index[
    coverage.eq(12).all(axis=1)
]

# Analyst-selected screening threshold, NOT an official BTS rule.
# Require at least 10,000 reported flights in EACH year.
MIN_ANNUAL_FLIGHTS = 10_000

flight_volume = airport_year.pivot(
    index="airport",
    columns="year",
    values="reported_flights"
).reindex(columns=[2023, 2024])

eligible_airports = flight_volume.index[
    flight_volume.ge(MIN_ANNUAL_FLIGHTS).all(axis=1)
    & flight_volume.index.isin(comparable_airports)
]

rates = (
    airport_year[
        airport_year["airport"].isin(eligible_airports)
    ]
    .pivot(
        index="airport",
        columns="year",
        values="late_minutes_per_100_flights"
    )
    .rename(columns={
        2023: "rate_2023",
        2024: "rate_2024"
    })
    .reset_index()
)

rates.columns.name = None
rates["change_minutes_per_100_flights"] = (
    rates["rate_2024"] - rates["rate_2023"]
)

rates = rates.merge(
    airport_names,
    on="airport",
    how="left",
    validate="one_to_one"
).sort_values(
    "change_minutes_per_100_flights",
    ascending=False
)

# -------------------------------------------------------------
# 5. Save complete outputs and ranking assumptions.
# -------------------------------------------------------------

airport_year["source_snapshot"] = bts_raw_path.name
monthly_late["source_snapshot"] = bts_raw_path.name

rates["minimum_annual_flights"] = MIN_ANNUAL_FLIGHTS
rates["required_months_each_year"] = 12
rates["source_snapshot"] = bts_raw_path.name

airport_year.to_csv(
    PROJECT / f"data/processed/bts_airport_late_delays_{bts_run_id}.csv",
    index=False
)
monthly_late.to_csv(
    PROJECT / f"data/processed/bts_monthly_late_delays_{bts_run_id}.csv",
    index=False
)
rates.to_csv(
    PROJECT / f"reports/airport_late_delay_rate_changes_{bts_run_id}.csv",
    index=False
)

# -------------------------------------------------------------
# 6. Display the investigation priorities.
# -------------------------------------------------------------

top_impact = (
    airport_year[airport_year["year"].eq(2024)]
    .nlargest(10, "late_aircraft_minutes")
)

print("A. HIGHEST TOTAL LATE-AIRCRAFT DELAY MINUTES — 2024")
display(
    top_impact[[
        "airport",
        "airport_name",
        "months_present",
        "reported_flights",
        "late_aircraft_minutes",
        "share_of_year_late_minutes_pct",
        "late_minutes_per_100_flights"
    ]].style.format({
        "reported_flights": "{:,.0f}",
        "late_aircraft_minutes": "{:,.0f}",
        "share_of_year_late_minutes_pct": "{:.2f}%",
        "late_minutes_per_100_flights": "{:,.2f}"
    })
)

print(
    "\nB. LARGEST INCREASES IN LATE-AIRCRAFT MINUTES PER 100 FLIGHTS"
)
print(
    f"Scope: all 24 months present and at least "
    f"{MIN_ANNUAL_FLIGHTS:,} flights in each year."
)

worsening = rates[
    rates["change_minutes_per_100_flights"].gt(0)
].head(10)

display(
    worsening[[
        "airport", "airport_name",
        "rate_2023", "rate_2024",
        "change_minutes_per_100_flights"
    ]].style.format({
        "rate_2023": "{:,.2f}",
        "rate_2024": "{:,.2f}",
        "change_minutes_per_100_flights": "{:+,.2f}"
    })
)

print("\nC. HIGHEST MONTHLY LATE-AIRCRAFT MINUTES PER 100 FLIGHTS")
display(
    monthly_late.nlargest(
        6, "late_minutes_per_100_flights"
    )[[
        "period", "reported_flights",
        "late_aircraft_minutes", "late_minutes_per_100_flights"
    ]].style.format({
        "period": lambda value: value.strftime("%Y-%m"),
        "reported_flights": "{:,.0f}",
        "late_aircraft_minutes": "{:,.0f}",
        "late_minutes_per_100_flights": "{:,.2f}"
    })
)

print("\nAirport and monthly aggregation reconciliation: PASSED")
print(f"Airports eligible for comparison: {len(eligible_airports)}")

logger.info("Airport and monthly late-aircraft analysis completed.")

## S14 — PostgreSQL installation and connection

Original code position 14; existing inline CELL labels may differ.


In [ ]:

import os
import sys
import secrets
import subprocess

def run_command(arguments, input_text=None):
    result = subprocess.run(
        arguments,
        input=input_text,
        text=True,
        capture_output=True,
        env={**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    )

    if result.returncode != 0:
        # Avoid exposing credentials that may appear in SQL error details.
        if input_text is None:
            print(result.stderr[-3000:])
        raise RuntimeError(
            f"Setup command failed: {arguments[0]}. "
            "Stop here and share the error."
        )

    return result.stdout.strip()

print("1/4 Installing PostgreSQL and Python connectors...")

run_command(["apt-get", "update", "-qq"])
run_command([
    "apt-get", "install", "-y", "-qq",
    "postgresql", "postgresql-client"
])
run_command([
    sys.executable, "-m", "pip", "install", "-q",
    "sqlalchemy", "psycopg2-binary"
])

# Detect the PostgreSQL cluster created by the package installer.
clusters = run_command(["pg_lsclusters", "--no-header"])
cluster_rows = [
    line.split() for line in clusters.splitlines() if line.strip()
]

main_clusters = [row for row in cluster_rows if row[1] == "main"]

if len(main_clusters) != 1:
    raise RuntimeError(
        "Expected one PostgreSQL main cluster. "
        "Share the output of !pg_lsclusters before continuing."
    )

cluster = main_clusters[0]
PG_VERSION = cluster[0]
PG_CLUSTER = cluster[1]
PG_PORT = int(cluster[2])

print("2/4 Starting the PostgreSQL server...")

if not cluster[3].startswith("online"):
    run_command([
        "pg_ctlcluster", PG_VERSION, PG_CLUSTER, "start"
    ])

# Administrative commands use the local postgres operating-system user.
admin_command = [
    "runuser", "-u", "postgres", "--",
    "psql",
    "-X",
    "-p", str(PG_PORT),
    "-d", "postgres",
    "-v", "ON_ERROR_STOP=1",
    "-t", "-A"
]

DB_NAME = "southwest_analytics"
DB_USER = "southwest_analyst"

# Generated in memory, never printed or saved in the notebook output.
# Hex characters are safe for the controlled SQL literal below.
DB_PASSWORD = secrets.token_hex(24)

print("3/4 Creating the project role and database if needed...")

role_exists = run_command(
    admin_command,
    f"SELECT 1 FROM pg_roles WHERE rolname = '{DB_USER}';"
)

if role_exists != "1":
    run_command(
        admin_command,
        f"CREATE ROLE {DB_USER} LOGIN PASSWORD '{DB_PASSWORD}';"
    )
else:
    # Refresh credentials for this runtime without deleting the role.
    run_command(
        admin_command,
        f"ALTER ROLE {DB_USER} WITH LOGIN PASSWORD '{DB_PASSWORD}';"
    )

database_exists = run_command(
    admin_command,
    f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}';"
)

if database_exists != "1":
    run_command(
        admin_command,
        f"CREATE DATABASE {DB_NAME} OWNER {DB_USER};"
    )

print("4/4 Testing the Python-to-PostgreSQL connection...")

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from IPython.display import display

# Dispose of an earlier connection pool if this cell is rerun.
if "engine" in globals():
    engine.dispose()

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host="127.0.0.1",
    port=PG_PORT,
    database=DB_NAME
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True,
    connect_args={"connect_timeout": 10}
)

with engine.connect() as connection:
    connection_test = pd.read_sql_query(
        text("""
            SELECT
                current_database() AS database_name,
                current_user AS connected_user,
                current_setting('server_version') AS postgres_version,
                'Connection successful' AS status;
        """),
        connection
    )

display(connection_test)

print("\nPostgreSQL is ready.")
print("Database:", DB_NAME)
print("Connection is local to this Colab runtime.")
print("Your validated CSV files have not been loaded yet.")

## S15 — Validated BTS load and backup — pinned input

Original code position 15; existing inline CELL labels may differ.


In [ ]:
# CELL 14 — Load validated BTS data into PostgreSQL and back it up
# Requires the recovery/setup cell and successful Cell 13.

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import subprocess

import numpy as np
import pandas as pd
from sqlalchemy import text
from IPython.display import display

PROJECT = Path("/content/drive/MyDrive/southwest_analytics")
bts_run_id = "20261113T180223779285Z"

csv_path = (
    PROJECT
    / f"data/processed/bts_arrivals_2023_2024_{bts_run_id}.csv"
)

if not csv_path.exists():
    raise FileNotFoundError(f"Validated dataset not found: {csv_path}")

load_df = pd.read_csv(csv_path)

integer_columns = [
    "year", "month",
    "arr_flights", "arr_del15", "arr_cancelled", "arr_diverted"
]

minute_columns = [
    "arr_delay", "carrier_delay", "weather_delay",
    "nas_delay", "security_delay", "late_aircraft_delay"
]

cause_count_columns = [
    "carrier_ct", "weather_ct", "nas_ct",
    "security_ct", "late_aircraft_ct"
]

text_columns = [
    "carrier", "carrier_name", "airport", "airport_name",
    "source_snapshot"
]

data_columns = (
    integer_columns
    + text_columns
    + cause_count_columns
    + minute_columns
)

missing = sorted(set(data_columns) - set(load_df.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

if load_df[data_columns].isna().any().any():
    raise ValueError("Required source fields contain missing values.")

# Validate integer-valued fields before converting.
for column in integer_columns + minute_columns:
    values = pd.to_numeric(load_df[column], errors="raise")

    if not np.isfinite(values).all() or not values.mod(1).eq(0).all():
        raise ValueError(f"{column} contains invalid integer values.")

    load_df[column] = values.astype("int64")

for column in cause_count_columns:
    values = pd.to_numeric(load_df[column], errors="raise")

    if not np.isfinite(values).all() or values.lt(0).any():
        raise ValueError(f"{column} contains invalid cause counts.")

    load_df[column] = values

primary_key = ["year", "month", "carrier", "airport"]

if load_df.duplicated(primary_key).any():
    raise ValueError("Duplicate source primary keys detected.")

if (
    not load_df["carrier"].eq("WN").all()
    or not load_df["year"].isin([2023, 2024]).all()
):
    raise ValueError("Unexpected carrier or reporting years.")

# Hash identifies the exact processed file loaded.
source_csv_sha256 = hashlib.sha256(csv_path.read_bytes()).hexdigest()
load_df["source_csv_sha256"] = source_csv_sha256
data_columns.append("source_csv_sha256")

records = load_df[data_columns].to_dict(orient="records")

# SQL constraints protect the database independently of Python.
minute_definitions = ",\n".join(
    f"{column} BIGINT NOT NULL CHECK ({column} >= 0)"
    for column in minute_columns
)

cause_definitions = ",\n".join(
    f"{column} NUMERIC(14,2) NOT NULL CHECK ({column} >= 0)"
    for column in cause_count_columns
)

create_table_sql = f"""
CREATE TABLE IF NOT EXISTS analytics.airport_month_operations (
    year SMALLINT NOT NULL CHECK (year BETWEEN 2000 AND 2100),
    month SMALLINT NOT NULL CHECK (month BETWEEN 1 AND 12),

    carrier TEXT NOT NULL CHECK (carrier <> ''),
    carrier_name TEXT NOT NULL,
    airport TEXT NOT NULL CHECK (airport ~ '^[A-Z]{{3}}$'),
    airport_name TEXT NOT NULL,

    arr_flights BIGINT NOT NULL CHECK (arr_flights >= 0),
    arr_del15 BIGINT NOT NULL CHECK (arr_del15 >= 0),
    arr_cancelled BIGINT NOT NULL CHECK (arr_cancelled >= 0),
    arr_diverted BIGINT NOT NULL CHECK (arr_diverted >= 0),

    {cause_definitions},
    {minute_definitions},

    source_snapshot TEXT NOT NULL,
    source_csv_sha256 TEXT NOT NULL,
    loaded_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,

    PRIMARY KEY (year, month, carrier, airport),

    CHECK (
        arr_del15 + arr_cancelled + arr_diverted <= arr_flights
    ),

    CHECK (
        arr_delay = carrier_delay + weather_delay + nas_delay
                    + security_delay + late_aircraft_delay
    ),

    CHECK (
        ABS(
            carrier_ct + weather_ct + nas_ct
            + security_ct + late_aircraft_ct - arr_del15
        ) <= 0.03
    )
);
"""

# Controlled column names defined above, not external input.
column_list = ", ".join(data_columns)
parameter_list = ", ".join(f":{column}" for column in data_columns)

update_columns = [
    column for column in data_columns if column not in primary_key
]

updates = ", ".join(
    f"{column} = EXCLUDED.{column}" for column in update_columns
)

upsert_sql = text(f"""
INSERT INTO analytics.airport_month_operations ({column_list})
VALUES ({parameter_list})
ON CONFLICT (year, month, carrier, airport)
DO UPDATE SET
    {updates},
    loaded_at = CURRENT_TIMESTAMP;
""")

reconciliation_sql = text("""
SELECT
    year,
    COUNT(*) AS row_count,
    SUM(arr_flights) AS reported_flights,
    SUM(arr_del15) AS delayed_arrivals,
    SUM(arr_cancelled) AS cancelled_flights,
    SUM(arr_diverted) AS diverted_flights,
    SUM(arr_delay) AS delay_minutes,
    SUM(late_aircraft_delay) AS late_aircraft_minutes
FROM analytics.airport_month_operations
WHERE carrier = 'WN'
  AND year IN (2023, 2024)
GROUP BY year
ORDER BY year;
""")

expected = (
    load_df.groupby("year")
    .agg(
        row_count=("airport", "size"),
        reported_flights=("arr_flights", "sum"),
        delayed_arrivals=("arr_del15", "sum"),
        cancelled_flights=("arr_cancelled", "sum"),
        diverted_flights=("arr_diverted", "sum"),
        delay_minutes=("arr_delay", "sum"),
        late_aircraft_minutes=("late_aircraft_delay", "sum")
    )
    .sort_index()
)

# Commit only if loading AND reconciliation succeed.
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS analytics;"))
    connection.execute(text(create_table_sql))
    connection.execute(upsert_sql, records)

    sql_totals = pd.read_sql_query(reconciliation_sql, connection)
    actual = sql_totals.set_index("year").sort_index()

    if actual.index.tolist() != expected.index.tolist():
        raise ValueError("SQL reporting years differ from the source.")

    verification_rows = []

    for year in expected.index:
        for metric in expected.columns:
            source_value = int(expected.loc[year, metric])
            database_value = int(actual.loc[year, metric])

            verification_rows.append({
                "year": int(year),
                "metric": metric,
                "source_value": source_value,
                "database_value": database_value,
                "status": (
                    "PASS" if source_value == database_value else "FAIL"
                )
            })

    sql_validation = pd.DataFrame(verification_rows)

    if not sql_validation["status"].eq("PASS").all():
        display(sql_validation)
        raise ValueError(
            "SQL reconciliation failed. Database changes were rolled back."
        )

print("SQL LOAD COMMITTED")
print(f"Source records loaded or updated: {len(records):,}")
print("Table: analytics.airport_month_operations")

print("\nTOTALS CALCULATED BY POSTGRESQL")
display(sql_totals)

print("\nSOURCE-TO-SQL VALIDATION")
display(sql_validation)

load_timestamp = datetime.now(timezone.utc)
load_id = load_timestamp.strftime("%Y%m%dT%H%M%S%fZ")

sql_validation.to_csv(
    PROJECT / f"reports/validation/postgres_load_checks_{load_id}.csv",
    index=False
)

backup_directory = PROJECT / "backups/postgresql"
backup_directory.mkdir(parents=True, exist_ok=True)

backup_path = backup_directory / f"southwest_analytics_{load_id}.dump"
partial_path = backup_path.with_suffix(".dump.partial")

# Match pg_dump's major version to the server installed in Cell 13.
pg_dump_path = f"/usr/lib/postgresql/{PG_VERSION}/bin/pg_dump"

backup_result = subprocess.run(
    [
        pg_dump_path,
        "--host=127.0.0.1",
        f"--port={PG_PORT}",
        f"--username={DB_USER}",
        f"--dbname={DB_NAME}",
        "--format=custom",
        "--no-owner",
        "--no-privileges",
        f"--file={partial_path}"
    ],
    env={**os.environ, "PGPASSWORD": DB_PASSWORD},
    capture_output=True,
    text=True
)

if backup_result.returncode != 0:
    partial_path.unlink(missing_ok=True)
    print(backup_result.stderr[-2000:])
    raise RuntimeError(
        "The SQL load succeeded, but the database backup failed."
    )

partial_path.replace(backup_path)

backup_metadata = {
    "database": DB_NAME,
    "created_at_utc": load_timestamp.isoformat(),
    "postgres_major_version": str(PG_VERSION),
    "source_csv": str(csv_path.relative_to(PROJECT)),
    "source_csv_sha256": source_csv_sha256,
    "source_row_count": len(load_df),
    "backup_file": backup_path.name,
    "backup_sha256": hashlib.sha256(backup_path.read_bytes()).hexdigest(),
    "restore_test_status": "NOT_YET_TESTED"
}

backup_path.with_suffix(".manifest.json").write_text(
    json.dumps(backup_metadata, indent=2),
    encoding="utf-8"
)

print("\nDatabase backup saved:", backup_path)
print("Backup restoration has not yet been tested.")

## S16 — Annual SQL analysis

Original code position 16; existing inline CELL labels may differ.


In [ ]:
# CELL 15 — Annual performance analysis using PostgreSQL
# Requires successful Cells 13 and 14.

annual_performance_sql = """
WITH annual_totals AS (
    SELECT
        year,
        SUM(arr_flights) AS reported_flights,
        SUM(arr_del15) AS delayed_arrivals,
        SUM(arr_cancelled) AS cancelled_flights,
        SUM(arr_diverted) AS diverted_flights,
        SUM(arr_delay) AS delay_minutes
    FROM analytics.airport_month_operations
    WHERE carrier = 'WN'
      AND year IN (2023, 2024)
    GROUP BY year
),
annual_rates AS (
    SELECT
        *,
        100.0 * cancelled_flights
            / NULLIF(reported_flights, 0)
            AS cancellation_rate_pct,

        100.0 * delayed_arrivals
            / NULLIF(reported_flights, 0)
            AS delayed_flight_share_pct,

        100.0 * (
            reported_flights
            - delayed_arrivals
            - cancelled_flights
            - diverted_flights
        ) / NULLIF(reported_flights, 0)
            AS on_time_flight_share_pct,

        1.0 * delay_minutes
            / NULLIF(delayed_arrivals, 0)
            AS avg_delay_minutes_per_delayed_arrival

    FROM annual_totals
)
SELECT
    *,
    cancellation_rate_pct
        - LAG(cancellation_rate_pct) OVER (ORDER BY year)
        AS cancellation_change_pp,

    delayed_flight_share_pct
        - LAG(delayed_flight_share_pct) OVER (ORDER BY year)
        AS delayed_share_change_pp,

    on_time_flight_share_pct
        - LAG(on_time_flight_share_pct) OVER (ORDER BY year)
        AS on_time_share_change_pp,

    avg_delay_minutes_per_delayed_arrival
        - LAG(avg_delay_minutes_per_delayed_arrival)
            OVER (ORDER BY year)
        AS average_delay_change_minutes

FROM annual_rates
ORDER BY year;
"""

with engine.connect() as connection:
    sql_annual_performance = pd.read_sql_query(
        text(annual_performance_sql),
        connection
    )

# -------------------------------------------------------------
# Independently calculate matching metrics from the saved CSV.
# -------------------------------------------------------------

reference = pd.read_csv(
    PROJECT
    / f"data/processed/bts_arrivals_2023_2024_{bts_run_id}.csv"
)

python_annual = (
    reference.groupby("year")[
        ["arr_flights", "arr_del15", "arr_cancelled",
         "arr_diverted", "arr_delay"]
    ]
    .sum()
    .sort_index()
)

flights = python_annual["arr_flights"].where(
    python_annual["arr_flights"].ne(0)
)
delayed = python_annual["arr_del15"].where(
    python_annual["arr_del15"].ne(0)
)

python_annual["cancellation_rate_pct"] = (
    python_annual["arr_cancelled"] / flights * 100
)
python_annual["delayed_flight_share_pct"] = (
    python_annual["arr_del15"] / flights * 100
)
python_annual["on_time_flight_share_pct"] = (
    (
        python_annual["arr_flights"]
        - python_annual["arr_del15"]
        - python_annual["arr_cancelled"]
        - python_annual["arr_diverted"]
    ) / flights * 100
)
python_annual["avg_delay_minutes_per_delayed_arrival"] = (
    python_annual["arr_delay"] / delayed
)

change_mapping = {
    "cancellation_change_pp": "cancellation_rate_pct",
    "delayed_share_change_pp": "delayed_flight_share_pct",
    "on_time_share_change_pp": "on_time_flight_share_pct",
    "average_delay_change_minutes":
        "avg_delay_minutes_per_delayed_arrival"
}

for change_column, metric_column in change_mapping.items():
    python_annual[change_column] = python_annual[metric_column].diff()

sql_indexed = sql_annual_performance.set_index("year").sort_index()

if sql_indexed.index.tolist() != python_annual.index.tolist():
    raise ValueError("SQL and Python reporting years do not match.")

metrics = list(change_mapping.values()) + list(change_mapping.keys())
checks = []

for year in python_annual.index:
    for metric in metrics:
        python_value = float(python_annual.loc[year, metric])
        sql_value = float(sql_indexed.loc[year, metric])

        passed = np.isclose(
            python_value,
            sql_value,
            rtol=0,
            atol=0.000001,
            equal_nan=True
        )

        checks.append({
            "year": int(year),
            "metric": metric,
            "python_value": python_value,
            "sql_value": sql_value,
            "status": "PASS" if passed else "FAIL"
        })

kpi_validation = pd.DataFrame(checks)

kpi_validation.to_csv(
    PROJECT / f"reports/validation/sql_annual_kpi_checks_{bts_run_id}.csv",
    index=False
)

if not kpi_validation["status"].eq("PASS").all():
    display(kpi_validation)
    raise ValueError("SQL metrics do not match the Python calculations.")

# Save the SQL itself as a reusable project asset.
sql_directory = PROJECT / "sql"
sql_directory.mkdir(parents=True, exist_ok=True)

sql_path = sql_directory / "01_annual_operational_performance.sql"
sql_path.write_text(annual_performance_sql.strip(), encoding="utf-8")

sql_annual_performance.to_csv(
    PROJECT / f"reports/sql_annual_performance_{bts_run_id}.csv",
    index=False
)

print("ANNUAL PERFORMANCE — CALCULATED IN POSTGRESQL")

display(
    sql_annual_performance[[
        "year",
        "reported_flights",
        "cancellation_rate_pct",
        "delayed_flight_share_pct",
        "on_time_flight_share_pct",
        "avg_delay_minutes_per_delayed_arrival",
        "average_delay_change_minutes"
    ]].style.format({
        "reported_flights": "{:,.0f}",
        "cancellation_rate_pct": "{:.2f}%",
        "delayed_flight_share_pct": "{:.2f}%",
        "on_time_flight_share_pct": "{:.2f}%",
        "avg_delay_minutes_per_delayed_arrival": "{:.2f}",
        "average_delay_change_minutes": "{:+.2f}"
    }, na_rep="N/A")
)

print("\nSQL-to-Python validation: PASSED")
print(f"Checks passed: {len(kpi_validation)}")
print("SQL query saved:", sql_path)

## S17 — Monthly SQL view and dashboard export

Original code position 17; existing inline CELL labels may differ.


In [ ]:
monthly_view_sql = """
CREATE OR REPLACE VIEW analytics.v_monthly_operations AS
WITH monthly_totals AS (
    SELECT
        carrier,
        MAKE_DATE(year, month, 1) AS period,
        COUNT(DISTINCT airport) AS airports_reporting,
        SUM(arr_flights) AS reported_flights,
        SUM(arr_del15) AS delayed_arrivals,
        SUM(arr_cancelled) AS cancelled_flights,
        SUM(arr_diverted) AS diverted_flights,
        SUM(arr_delay) AS delay_minutes,
        SUM(late_aircraft_delay) AS late_aircraft_minutes
    FROM analytics.airport_month_operations
    GROUP BY carrier, year, month
)
SELECT
    *,
    reported_flights
        - delayed_arrivals
        - cancelled_flights
        - diverted_flights AS on_time_flights,

    100.0 * cancelled_flights
        / NULLIF(reported_flights, 0)
        AS cancellation_rate_pct,

    100.0 * delayed_arrivals
        / NULLIF(reported_flights, 0)
        AS delayed_flight_share_pct,

    100.0 * (
        reported_flights
        - delayed_arrivals
        - cancelled_flights
        - diverted_flights
    ) / NULLIF(reported_flights, 0)
        AS on_time_flight_share_pct,

    1.0 * delay_minutes
        / NULLIF(delayed_arrivals, 0)
        AS avg_delay_minutes_per_delayed_arrival,

    100.0 * late_aircraft_minutes
        / NULLIF(reported_flights, 0)
        AS late_minutes_per_100_flights

FROM monthly_totals;
"""

monthly_query = text("""
SELECT *
FROM analytics.v_monthly_operations
WHERE carrier = 'WN'
  AND period >= DATE '2023-01-01'
  AND period < DATE '2025-01-01'
ORDER BY period;
""")

# Independently build monthly reference totals from the saved source CSV.
reference = pd.read_csv(
    PROJECT
    / f"data/processed/bts_arrivals_2023_2024_{bts_run_id}.csv"
)

reference["period"] = pd.to_datetime({
    "year": reference["year"],
    "month": reference["month"],
    "day": 1
})

python_monthly = (
    reference.groupby("period")
    .agg(
        airports_reporting=("airport", "nunique"),
        reported_flights=("arr_flights", "sum"),
        delayed_arrivals=("arr_del15", "sum"),
        cancelled_flights=("arr_cancelled", "sum"),
        diverted_flights=("arr_diverted", "sum"),
        delay_minutes=("arr_delay", "sum"),
        late_aircraft_minutes=("late_aircraft_delay", "sum")
    )
    .sort_index()
)

python_monthly["on_time_flights"] = (
    python_monthly["reported_flights"]
    - python_monthly["delayed_arrivals"]
    - python_monthly["cancelled_flights"]
    - python_monthly["diverted_flights"]
)

rate_definitions = {
    "cancellation_rate_pct":
        ("cancelled_flights", "reported_flights", 100),
    "delayed_flight_share_pct":
        ("delayed_arrivals", "reported_flights", 100),
    "on_time_flight_share_pct":
        ("on_time_flights", "reported_flights", 100),
    "avg_delay_minutes_per_delayed_arrival":
        ("delay_minutes", "delayed_arrivals", 1),
    "late_minutes_per_100_flights":
        ("late_aircraft_minutes", "reported_flights", 100)
}

for metric, (numerator, denominator, scale) in rate_definitions.items():
    python_monthly[metric] = (
        python_monthly[numerator]
        / python_monthly[denominator].where(
            python_monthly[denominator].ne(0)
        )
        * scale
    )

# Create and test within one transaction.
# A failed check rolls back the view change.
with engine.begin() as connection:
    connection.execute(text(monthly_view_sql))

    sql_monthly = pd.read_sql_query(monthly_query, connection)
    sql_monthly["period"] = pd.to_datetime(sql_monthly["period"])

    if sql_monthly.duplicated(["carrier", "period"]).any():
        raise ValueError("Duplicate carrier-month rows in the view.")

    actual = sql_monthly.set_index("period").sort_index()
    expected_periods = pd.date_range(
        "2023-01-01", "2024-12-01", freq="MS"
    )

    if not actual.index.equals(expected_periods):
        raise ValueError("The SQL view does not contain all 24 months.")

    if not actual.index.equals(python_monthly.index):
        raise ValueError("SQL and source periods do not match.")

    validation_rows = []

    for metric in python_monthly.columns:
        matches = np.isclose(
            actual[metric].to_numpy(dtype=float),
            python_monthly[metric].to_numpy(dtype=float),
            rtol=0,
            atol=0.000001,
            equal_nan=True
        )

        validation_rows.append({
            "metric": metric,
            "months_checked": len(matches),
            "mismatched_months": int((~matches).sum()),
            "status": "PASS" if matches.all() else "FAIL"
        })

    monthly_validation = pd.DataFrame(validation_rows)

    if not monthly_validation["status"].eq("PASS").all():
        display(monthly_validation)
        raise ValueError("Monthly view failed source reconciliation.")

# Save the definition so the view can be recreated after a runtime reset.
sql_directory = PROJECT / "sql"
sql_directory.mkdir(parents=True, exist_ok=True)

view_path = sql_directory / "02_monthly_operations_view.sql"
view_path.write_text(monthly_view_sql.strip(), encoding="utf-8")

export_path = (
    PROJECT / f"data/processed/sql_monthly_operations_{bts_run_id}.csv"
)
sql_monthly.to_csv(export_path, index=False)

monthly_validation.to_csv(
    PROJECT / f"reports/validation/sql_monthly_view_checks_{bts_run_id}.csv",
    index=False
)

print("MONTHLY REPORTING VIEW — MOST RECENT SIX MONTHS")
display(
    sql_monthly.tail(6)[[
        "period",
        "airports_reporting",
        "reported_flights",
        "cancellation_rate_pct",
        "delayed_flight_share_pct",
        "on_time_flight_share_pct",
        "avg_delay_minutes_per_delayed_arrival"
    ]].style.format({
        "period": lambda value: value.strftime("%Y-%m"),
        "reported_flights": "{:,.0f}",
        "cancellation_rate_pct": "{:.2f}%",
        "delayed_flight_share_pct": "{:.2f}%",
        "on_time_flight_share_pct": "{:.2f}%",
        "avg_delay_minutes_per_delayed_arrival": "{:.2f}"
    })
)

print("\nMONTHLY VIEW VALIDATION")
display(monthly_validation)

print("\nView created: analytics.v_monthly_operations")
print("SQL definition saved:", view_path)
print("Dashboard-ready CSV saved:", export_path)
print("The earlier database backup does not yet include this new view.")

## S18 — Optional dashboard CSV download

Original code position 18; existing inline CELL labels may differ.


In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/southwest_analytics/"
    "data/processed/"
    "sql_monthly_operations_20261113T180223779285Z.csv"
)

## S19 — Optional Drive mount

Original code position 19; existing inline CELL labels may differ.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## S20 — Optional financial file downloads

Original code position 20; existing inline CELL labels may differ.


In [ ]:
from google.colab import files
from pathlib import Path

folder = Path("/content/drive/MyDrive/southwest_analytics/data/processed")

filenames = [
    "financials_2024_20261113T174840400549Z.csv",
    "expense_comparison_2023_2024_20261113T174840400549Z.csv",
]

for name in filenames:
    path = folder / name
    if path.is_file():
        files.download(str(path))
    else:
        print(f"File not found: {name}")

## S21 — Optional financial CSV inspection

Original code position 21; existing inline CELL labels may differ.


In [ ]:
import pandas as pd
from pathlib import Path

folder = Path("/content/drive/MyDrive/southwest_analytics/data/processed")

for name in [
    "financials_2024_20261113T174840400549Z.csv",
    "expense_comparison_2023_2024_20261113T174840400549Z.csv",
]:
    print(f"\nFILE: {name}")
    path = folder / name
    if not path.exists():
        print("File not found")
        continue

    df = pd.read_csv(path)
    print("COLUMNS:", df.columns.tolist())
    print(df.to_string(index=False))

## S22 — Monthly forecast input inspection

Original code position 22; existing inline CELL labels may differ.


In [ ]:
from pathlib import Path
import pandas as pd

path = Path(
    "/content/drive/MyDrive/southwest_analytics/data/processed/"
    "sql_monthly_operations_20261113T180223779285Z.csv"
)

if not path.exists():
    raise FileNotFoundError(
        "Monthly SQL export not found. Mount Google Drive and check the path."
    )

monthly = pd.read_csv(path)
print("Columns:", monthly.columns.tolist())
print("Rows:", len(monthly))
print(monthly.head(3).to_string(index=False))

## S23 — 2024 forecasting backtest

Original code position 23; existing inline CELL labels may differ.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import calendar
import hashlib
import json
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/southwest_analytics")
SOURCE = PROJECT / "data/processed/sql_monthly_operations_20261113T180223779285Z.csv"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = PROJECT / "reports/forecasting" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

# 1. Load and validate the saved source.
df = pd.read_csv(SOURCE, dtype={"carrier": "string"})
required = {"carrier", "period", "reported_flights"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")

df["period"] = pd.to_datetime(df["period"], errors="raise")
df["reported_flights"] = pd.to_numeric(
    df["reported_flights"], errors="raise"
)

expected = pd.date_range("2023-01-01", "2024-12-01", freq="MS")
values = df["reported_flights"].to_numpy(dtype=float)

checks = {
    "Exactly 24 records": len(df) == 24,
    "Only carrier WN": df["carrier"].notna().all()
                       and df["carrier"].eq("WN").all(),
    "Unique monthly records": not df["period"].duplicated().any(),
    "Exact 2023–2024 month coverage":
        set(df["period"]) == set(expected),
    "Flight counts are finite positive integers":
        bool(np.isfinite(values).all()
             and (values > 0).all()
             and (values == np.floor(values)).all()),
}

validation = pd.DataFrame([
    {"check": name, "status": "PASS" if passed else "FAIL"}
    for name, passed in checks.items()
])
validation.to_csv(OUT / "input_validation.csv", index=False)

if not all(checks.values()):
    display(validation)
    raise ValueError("Forecast input validation failed.")

df = df.sort_values("period")
train = df[df["period"].dt.year.eq(2023)].copy()
test = df[df["period"].dt.year.eq(2024)].copy()

# 2. Freeze forecasts using ONLY 2023 observations.
# Forecast origin: end of December 2023.
# Benchmark A: same month last year.
# Benchmark B: same month last year's daily flight rate,
# adjusted for calendar days (including February 2024).
forecast = pd.DataFrame({
    "period": pd.date_range("2024-01-01", "2024-12-01", freq="MS"),
    "forecast_origin": "2023-12-31",
    "source_period": train["period"].to_numpy(),
    "prior_year_flights": train["reported_flights"].to_numpy(),
})

forecast["seasonal_naive"] = forecast["prior_year_flights"]

target_days = np.array([
    calendar.monthrange(2024, month)[1] for month in range(1, 13)
])
source_days = train["period"].dt.days_in_month.to_numpy()

forecast["calendar_adjusted_seasonal"] = np.rint(
    forecast["prior_year_flights"] / source_days * target_days
).astype("int64")

# Save the forecasts before attaching evaluation actuals.
forecast.to_csv(OUT / "frozen_forecasts_2024.csv", index=False)

# 3. Evaluate against held-out 2024 actuals.
evaluation = forecast.merge(
    test[["period", "reported_flights"]].rename(
        columns={"reported_flights": "actual_flights"}
    ),
    on="period",
    how="left",
    validate="one_to_one"
)

models = ["seasonal_naive", "calendar_adjusted_seasonal"]
rows = []
details = []

for model in models:
    actual = evaluation["actual_flights"].to_numpy(dtype=float)
    predicted = evaluation[model].to_numpy(dtype=float)

    # Positive error means the forecast was higher than actual.
    error = predicted - actual
    absolute_error = np.abs(error)

    rows.append({
        "model": model,
        "months_tested": len(actual),
        "MAE_flights": absolute_error.mean(),
        "RMSE_flights": np.sqrt(np.mean(error ** 2)),
        "MAPE_pct": np.mean(absolute_error / actual) * 100,
        "WAPE_pct": absolute_error.sum() / actual.sum() * 100,
        "bias_pct": error.sum() / actual.sum() * 100,
        "forecast_total": predicted.sum(),
        "actual_total": actual.sum(),
        "forecast_minus_actual_total": error.sum(),
    })

    details.append(pd.DataFrame({
        "period": evaluation["period"],
        "model": model,
        "actual_flights": actual,
        "forecast_flights": predicted,
        "forecast_minus_actual": error,
        "absolute_error": absolute_error,
        "absolute_percentage_error": absolute_error / actual * 100,
    }))

metrics = pd.DataFrame(rows)
monthly_errors = pd.concat(details, ignore_index=True)

metrics.to_csv(OUT / "forecast_accuracy.csv", index=False)
monthly_errors.to_csv(OUT / "monthly_forecast_errors.csv", index=False)

# 4. Save provenance and methodological limitations.
manifest = {
    "run_id": RUN_ID,
    "source_file": str(SOURCE),
    "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
    "target": "Monthly BTS-reported Southwest domestic arrival flight counts",
    "training_period": "2023-01 through 2023-12",
    "evaluation_period": "2024-01 through 2024-12",
    "forecast_origin": "2023-12-31",
    "models": models,
    "error_convention": "forecast minus actual; positive means overforecast",
    "limitations": [
        "Retrospective backtest using the saved source snapshot; "
        "not a reconstruction of data available in December 2023.",
        "Only one training year and one evaluation year.",
        "Calendar adjustment accounts for month length, not weekday mix.",
        "No schedule, capacity, airport exit, or external predictors used.",
        "2024 is the evaluation set, not an independent confirmation "
        "set for future model tuning.",
        "Forecasts are analyst benchmarks, not company guidance or budgets.",
        "No prediction intervals estimated."
    ]
}

(OUT / "forecast_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("FORECAST ACCURACY — HELD-OUT 2024")
display(metrics.round(2))

print("\nMONTHLY RESULTS — SEASONAL NAIVE")
display(
    monthly_errors[
        monthly_errors["model"].eq("seasonal_naive")
    ].round(2)
)

print("\nPositive bias = overforecast; negative bias = underforecast.")
print("MAE = average absolute monthly error, measured in flights.")
print("WAPE = total absolute monthly error / total actual flights.")
print(f"\nSaved forecasts, evaluation, validation and provenance:\n{OUT}")

## S24 — Backup prerequisites check

Original code position 24; existing inline CELL labels may differ.


In [ ]:
from pathlib import Path
import shutil
import subprocess
from google.colab import drive

drive.mount("/content/drive")

project = Path("/content/drive/MyDrive/southwest_analytics")
backup = project / (
    "backups/postgresql/"
    "southwest_analytics_20261114T054433356945Z.dump"
)

print("BACKUP CHECK")
print("Found:", backup.is_file())
if backup.is_file():
    print("Size:", f"{backup.stat().st_size:,}", "bytes")

print("\nPOSTGRESQL TOOLS")
for tool in ["psql", "pg_restore", "pg_lsclusters"]:
    print(f"{tool}: {shutil.which(tool) or 'NOT INSTALLED'}")

if shutil.which("pg_lsclusters"):
    result = subprocess.run(
        ["pg_lsclusters"],
        capture_output=True,
        text=True
    )
    print("\nPOSTGRESQL CLUSTERS")
    print(result.stdout or result.stderr)

if backup.is_file() and shutil.which("pg_restore"):
    result = subprocess.run(
        ["pg_restore", "--list", str(backup)],
        capture_output=True,
        text=True
    )
    print("\nBACKUP CONTENT CHECK")
    if result.returncode != 0:
        print(result.stderr)
    else:
        entries = [
            line for line in result.stdout.splitlines()
            if not line.startswith(";")
            and "airport_month_operations" in line
        ]
        print("\n".join(entries) or "Expected table not found in archive.")
        print("\nArchive listing succeeded; actual restoration is still pending.")

## S25 — Isolated database restore test

Original code position 25; existing inline CELL labels may differ.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess
import hashlib
import io
import json
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/southwest_analytics")
BACKUP = PROJECT / "backups/postgresql/southwest_analytics_20261114T054433356945Z.dump"
SOURCE = PROJECT / "data/processed/bts_arrivals_2023_2024_20261113T180223779285Z.csv"

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
test_db = f"southwest_restore_{run_id.lower()}"
report_dir = PROJECT / "reports/validation" / f"restore_{run_id}"
report_dir.mkdir(parents=True, exist_ok=True)

def postgres_command(arguments, **kwargs):
    return subprocess.run(
        ["sudo", "-u", "postgres", *arguments],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        **kwargs
    )

def require_success(result, step):
    if result.returncode:
        error = result.stderr.decode(errors="replace")
        raise RuntimeError(f"{step} failed:\n{error}")

manifest = {
    "run_id": run_id,
    "test_database": test_db,
    "backup": str(BACKUP),
    "source_csv": str(SOURCE),
    "status": "STARTED"
}

try:
    if not BACKUP.is_file() or not SOURCE.is_file():
        raise FileNotFoundError("Backup or validated BTS CSV is missing.")

    manifest["backup_sha256"] = hashlib.sha256(BACKUP.read_bytes()).hexdigest()
    manifest["source_sha256"] = hashlib.sha256(SOURCE.read_bytes()).hexdigest()

    print("1/3 Creating isolated test database...")
    result = postgres_command([
        "createdb", "-p", "5432", "--template=template0", test_db
    ])
    require_success(result, "Database creation")

    print("2/3 Restoring backup...")
    # Stream from Drive through stdin so postgres need not access Drive directly.
    with BACKUP.open("rb") as archive:
        result = postgres_command([
            "pg_restore", "-p", "5432",
            "--dbname", test_db,
            "--no-owner", "--no-privileges",
            "--exit-on-error", "--single-transaction"
        ], stdin=archive)

    (report_dir / "restore.log").write_text(
        result.stderr.decode(errors="replace"),
        encoding="utf-8"
    )
    require_success(result, "Archive restoration")

    print("3/3 Comparing restored data with validated CSV...")
    result = postgres_command([
        "psql", "-X", "-p", "5432", "-d", test_db,
        "-v", "ON_ERROR_STOP=1", "-c",
        "COPY analytics.airport_month_operations TO STDOUT WITH CSV HEADER"
    ])
    require_success(result, "Reading restored table")

    restored = pd.read_csv(io.StringIO(result.stdout.decode()))
    source = pd.read_csv(SOURCE)

    # Accommodate raw BTS names or reporting names used by the SQL table.
    columns = {
        "reported_flights": ["reported_flights", "arr_flights"],
        "delayed_arrivals": ["delayed_arrivals", "arr_del15"],
        "cancelled_flights": ["cancelled_flights", "arr_cancelled"],
        "diverted_flights": ["diverted_flights", "arr_diverted"],
        "delay_minutes": ["delay_minutes", "arr_delay"],
        "late_aircraft_minutes": ["late_aircraft_minutes", "late_aircraft_delay"],
    }

    def annual_totals(frame):
        data = pd.DataFrame({
            "year": pd.to_numeric(frame["year"], errors="raise")
        })
        for metric, options in columns.items():
            matches = [name for name in options if name in frame.columns]
            if len(matches) != 1:
                raise ValueError(
                    f"Cannot unambiguously map {metric}; columns: "
                    f"{frame.columns.tolist()}"
                )
            values = pd.to_numeric(frame[matches[0]], errors="raise")
            if not np.isfinite(values.to_numpy(dtype=float)).all():
                raise ValueError(f"Missing or non-finite values in {metric}")
            data[metric] = values
        totals = data.groupby("year").sum()
        totals["row_count"] = data.groupby("year").size()
        return totals.sort_index()

    expected = annual_totals(source)
    actual = annual_totals(restored)

    if set(expected.index) != {2023, 2024}:
        raise ValueError("Source CSV must cover exactly 2023 and 2024.")
    if not expected.index.equals(actual.index):
        raise ValueError("Restored years differ from the source.")

    checks = []
    for year in expected.index:
        for metric in expected.columns:
            source_value = expected.loc[year, metric]
            restored_value = actual.loc[year, metric]
            checks.append({
                "year": int(year),
                "metric": metric,
                "source_value": source_value,
                "restored_value": restored_value,
                "status": "PASS" if source_value == restored_value else "FAIL"
            })

    validation = pd.DataFrame(checks)
    validation.to_csv(report_dir / "restore_reconciliation.csv", index=False)
    display(validation)

    if not validation["status"].eq("PASS").all():
        raise ValueError("Restored totals do not match the source.")

    manifest["status"] = "PASS"
    manifest["checks_passed"] = len(validation)
    manifest["restored_rows"] = len(restored)

    print(f"\nRESTORE TEST PASSED: {len(validation)} checks")
    print(f"Restored rows: {len(restored):,}")
    print(f"Test database retained: {test_db}")
    print(f"Evidence saved: {report_dir}")
    print(
        "\nScope: table restoration and annual aggregate reconciliation. "
        "This does not verify every individual field or application permissions."
    )

except Exception as exc:
    manifest["status"] = "FAIL"
    manifest["error"] = str(exc)
    raise

finally:
    (report_dir / "restore_manifest.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8"
    )

## S26 — Monthly view recovery — historical database target

Original code position 26; existing inline CELL labels may differ.


In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import numpy as np
import io
import json

PROJECT = Path("/content/drive/MyDrive/southwest_analytics")
TEST_DB = "southwest_restore_20261115t184531406040z"
REPORT_DIR = PROJECT / (
    "reports/validation/restore_20261115T184531406040Z"
)
SQL_FILE = PROJECT / "sql/02_monthly_operations_view.sql"
CSV_FILE = PROJECT / (
    "data/processed/"
    "sql_monthly_operations_20261113T180223779285Z.csv"
)

def run_sql(sql):
    result = subprocess.run(
        [
            "sudo", "-u", "postgres",
            "psql", "-X", "-p", "5432", "-d", TEST_DB,
            "-v", "ON_ERROR_STOP=1"
        ],
        input=sql,
        text=True,
        capture_output=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout

if not SQL_FILE.is_file() or not CSV_FILE.is_file():
    raise FileNotFoundError("Saved view SQL or dashboard CSV is missing.")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Execute the saved view definition in the restored test database.
sql = SQL_FILE.read_text(encoding="utf-8")
restore_output = run_sql(sql)
(REPORT_DIR / "view_recreation.log").write_text(
    restore_output, encoding="utf-8"
)

csv_output = run_sql(
    "COPY (SELECT * FROM analytics.v_monthly_operations) "
    "TO STDOUT WITH CSV HEADER;"
)
restored = pd.read_csv(io.StringIO(csv_output))
expected = pd.read_csv(CSV_FILE)

keys = ["carrier", "period"]
for frame in (expected, restored):
    frame["period"] = pd.to_datetime(frame["period"], errors="raise")
    if frame[keys].isna().any().any():
        raise ValueError("Missing carrier or period.")
    if frame.duplicated(keys).any():
        raise ValueError("Duplicate carrier-month keys.")

if set(expected.columns) != set(restored.columns):
    raise ValueError("Restored view columns differ from dashboard CSV.")

expected = expected.set_index(keys).sort_index()
restored = restored.set_index(keys).sort_index()

if len(expected) != 24 or not expected.index.equals(restored.index):
    raise ValueError("Expected exactly 24 matching carrier-month records.")

checks = []
for column in expected.columns:
    left = pd.to_numeric(expected[column], errors="raise").to_numpy(float)
    right = pd.to_numeric(restored[column], errors="raise").to_numpy(float)

    finite = np.isfinite(left) & np.isfinite(right)

    # Allow only tiny decimal serialization differences in rate fields.
    is_rate = (
        column.endswith("_pct")
        or column in {
            "avg_delay_minutes_per_delayed_arrival",
            "late_minutes_per_100_flights"
        }
    )
    matches = finite & np.isclose(
        left, right, rtol=0, atol=0.000001 if is_rate else 0
    )

    checks.append({
        "metric": column,
        "months_checked": len(left),
        "mismatched_months": int((~matches).sum()),
        "status": "PASS" if matches.all() else "FAIL"
    })

validation = pd.DataFrame(checks)
validation.to_csv(
    REPORT_DIR / "restored_monthly_view_validation.csv", index=False
)
display(validation)

passed = validation["status"].eq("PASS").all()
(REPORT_DIR / "view_recovery_status.json").write_text(
    json.dumps({
        "test_database": TEST_DB,
        "status": "PASS" if passed else "FAIL",
        "months": len(restored),
        "metrics": len(validation),
        "method": "Saved SQL recreation followed by dashboard CSV comparison"
    }, indent=2),
    encoding="utf-8"
)

if not passed:
    raise ValueError("Monthly view recovery validation failed.")

print("\nMONTHLY REPORTING VIEW RECOVERY: PASSED")
print(f"Compared {len(restored)} months across {len(validation)} metrics.")
print(f"Evidence saved: {REPORT_DIR}")

## S27 — Project inventory

Original code position 27; existing inline CELL labels may differ.


In [ ]:
from pathlib import Path

project = Path("/content/drive/MyDrive/southwest_analytics")

for folder_name in ["sql", "docs", "logs", "reports/validation"]:
    folder = project / folder_name
    print(f"\n--- {folder_name} ---")
    if not folder.exists():
        print("Not present")
        continue

    for path in sorted(folder.rglob("*")):
        if path.is_file():
            print(path.relative_to(project))

print("\n--- Saved Python scripts and notebooks ---")
for pattern in ["*.py", "*.ipynb"]:
    for path in sorted(project.rglob(pattern)):
        print(path.relative_to(project))

In [ ]:
# SOUTHWEST ANALYTICS — REPEATABLE CHECKPOINT PIPELINE
# Self-contained: does not require variables from previous cells.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import subprocess
import shutil
import hashlib
import json
import io
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# ---------- Single configuration ----------
PROJECT = Path("/content/drive/MyDrive/southwest_analytics")

CONFIG = {
    "bts_csv": "data/processed/bts_arrivals_2023_2024_20261113T180223779285Z.csv",
    "database_backup": "backups/postgresql/southwest_analytics_20261114T054433356945Z.dump",
    "monthly_view_sql": "sql/02_monthly_operations_view.sql",
}

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT = PROJECT / "reports/pipeline_runs" / RUN_ID
OUTPUT.mkdir(parents=True, exist_ok=False)

TEST_DB = "southwest_pipeline_" + RUN_ID.lower()
MANIFEST = {
    "run_id": RUN_ID,
    "status": "RUNNING",
    "scope": "Saved BTS checkpoint, backup recovery, reporting and forecasting",
    "database": TEST_DB,
    "configuration": CONFIG,
    "stages_completed": [],
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
}

def save_manifest():
    (OUTPUT / "manifest.json").write_text(
        json.dumps(MANIFEST, indent=2), encoding="utf-8"
    )

def log(message):
    print(message)
    with (OUTPUT / "pipeline.log").open("a", encoding="utf-8") as handle:
        handle.write(
            datetime.now(timezone.utc).isoformat() + " " + message + "\n"
        )

def complete(stage):
    MANIFEST["stages_completed"].append(stage)
    save_manifest()

def command(arguments, input_bytes=None, stdin=None):
    result = subprocess.run(
        arguments,
        input=input_bytes,
        stdin=stdin,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.returncode:
        raise RuntimeError(
            f"{arguments[0]} failed:\n"
            + result.stderr.decode(errors="replace")[-5000:]
        )
    return result.stdout.decode(errors="replace")

def require(condition, message):
    if not condition:
        raise ValueError(message)

save_manifest()

try:
    # ---------- 1. Verify preserved inputs ----------
    log("1/6 Checking input files and recording hashes")
    paths = {key: PROJECT / value for key, value in CONFIG.items()}

    MANIFEST["input_sha256"] = {}
    for name, path in paths.items():
        require(path.is_file(), f"Missing input: {path}")
        MANIFEST["input_sha256"][name] = hashlib.sha256(
            path.read_bytes()
        ).hexdigest()

    source = pd.read_csv(paths["bts_csv"])

    measures = [
        "arr_flights", "arr_del15", "arr_cancelled", "arr_diverted",
        "arr_delay", "carrier_delay", "weather_delay", "nas_delay",
        "security_delay", "late_aircraft_delay",
        "carrier_ct", "weather_ct", "nas_ct",
        "security_ct", "late_aircraft_ct",
    ]
    keys = ["year", "month", "carrier", "airport"]
    required = keys + measures

    require(set(required).issubset(source.columns),
            "Required BTS columns are missing.")
    require(not source[required].isna().any().any(),
            "Required BTS values are missing.")

    for column in ["year", "month"] + measures:
        source[column] = pd.to_numeric(source[column], errors="raise")
        values = source[column].to_numpy(dtype=float)
        require(np.isfinite(values).all(), f"Non-finite values: {column}")
        require((values >= 0).all(), f"Negative values: {column}")

    for column in [
        "year", "month", "arr_flights", "arr_del15",
        "arr_cancelled", "arr_diverted"
    ]:
        values = source[column].to_numpy()
        require((values == np.floor(values)).all(),
                f"Non-integer counts: {column}")

    require(source["carrier"].eq("WN").all(), "Unexpected carrier.")
    require(source["airport"].astype(str).str.fullmatch(r"[A-Z]{3}").all(),
            "Invalid airport identifier.")
    require(source["month"].between(1, 12).all(), "Invalid month.")
    require(set(source["year"]) == {2023, 2024}, "Unexpected years.")
    require(not source.duplicated(keys).any(), "Duplicate airport-month keys.")
    require(len(source) == 2556, "Pinned dataset must contain 2,556 rows.")

    source["period"] = pd.to_datetime({
        "year": source["year"].astype(int),
        "month": source["month"].astype(int),
        "day": 1,
    })
    require(
        set(source["period"]) ==
        set(pd.date_range("2023-01-01", "2024-12-01", freq="MS")),
        "Month coverage does not match the requested 24 months."
    )

    require((
        source["arr_del15"] + source["arr_cancelled"]
        + source["arr_diverted"] <= source["arr_flights"]
    ).all(), "Flight categories exceed reported flights.")

    cause_minutes = [
        "carrier_delay", "weather_delay", "nas_delay",
        "security_delay", "late_aircraft_delay"
    ]
    cause_counts = [
        "carrier_ct", "weather_ct", "nas_ct",
        "security_ct", "late_aircraft_ct"
    ]
    require(np.allclose(
        source[cause_minutes].sum(axis=1),
        source["arr_delay"], rtol=0, atol=0.000001
    ), "Delay minutes fail reconciliation.")

    require(np.allclose(
        source[cause_counts].sum(axis=1),
        source["arr_del15"], rtol=0, atol=0.03
    ), "Attributed delay counts fail reconciliation.")

    complete("BTS checkpoint validation")

    # ---------- 2. Build independent Python reporting reference ----------
    log("2/6 Calculating monthly reference results")

    mapping = {
        "arr_flights": "reported_flights",
        "arr_del15": "delayed_arrivals",
        "arr_cancelled": "cancelled_flights",
        "arr_diverted": "diverted_flights",
        "arr_delay": "delay_minutes",
        "late_aircraft_delay": "late_aircraft_minutes",
    }

    grouped = source.groupby(["carrier", "period"], sort=True)
    reference = grouped[list(mapping)].sum().rename(columns=mapping)
    reference["airports_reporting"] = grouped["airport"].nunique()
    reference["on_time_flights"] = (
        reference["reported_flights"]
        - reference["delayed_arrivals"]
        - reference["cancelled_flights"]
        - reference["diverted_flights"]
    )

    flights = reference["reported_flights"].replace(0, np.nan)
    delayed = reference["delayed_arrivals"].replace(0, np.nan)

    for target, numerator in {
        "cancellation_rate_pct": "cancelled_flights",
        "delayed_flight_share_pct": "delayed_arrivals",
        "on_time_flight_share_pct": "on_time_flights",
        "late_minutes_per_100_flights": "late_aircraft_minutes",
    }.items():
        reference[target] = 100 * reference[numerator] / flights

    reference["avg_delay_minutes_per_delayed_arrival"] = (
        reference["delay_minutes"] / delayed
    )

    reference.reset_index().to_csv(
        OUTPUT / "python_monthly_reference.csv", index=False
    )
    complete("Python monthly reference")

    # ---------- 3. Ensure PostgreSQL is available ----------
    log("3/6 Preparing PostgreSQL")

    if not all(shutil.which(tool) for tool in [
        "psql", "pg_restore", "pg_lsclusters", "createdb"
    ]):
        command(["apt-get", "update", "-qq"])
        command([
            "apt-get", "install", "-y", "-qq",
            "postgresql", "postgresql-client"
        ])

    cluster_text = command(["pg_lsclusters", "--no-header"])
    clusters = [line.split() for line in cluster_text.splitlines()
                if line.strip()]
    require(bool(clusters), "No PostgreSQL cluster is available.")

    # Prefer version 16, which produced the saved archive.
    selected = next((c for c in clusters if c[0] == "16"), clusters[0])
    version, cluster, port, status = selected[:4]

    if status != "online":
        command(["pg_ctlcluster", version, cluster, "start"])

    MANIFEST["environment"]["postgresql_client"] = command(
        ["psql", "--version"]
    ).strip()
    MANIFEST["postgresql_port"] = port

    def pg(arguments, **kwargs):
        return command(
            ["sudo", "-u", "postgres", *arguments], **kwargs
        )

    def sql(statement):
        return pg(
            ["psql", "-X", "-q", "-p", port, "-d", TEST_DB,
             "-v", "ON_ERROR_STOP=1"],
            input_bytes=statement.encode("utf-8"),
        )

    complete("PostgreSQL availability")

    # ---------- 4. Restore and rebuild reporting ----------
    log("4/6 Restoring an isolated database and rebuilding the view")

    pg(["createdb", "-p", port, "--template=template0", TEST_DB])

    with paths["database_backup"].open("rb") as archive:
        pg([
            "pg_restore", "-p", port, "--dbname", TEST_DB,
            "--no-owner", "--no-privileges",
            "--exit-on-error", "--single-transaction",
        ], stdin=archive)

    restored_count = sql(
        "COPY (SELECT COUNT(*) AS rows "
        "FROM analytics.airport_month_operations) "
        "TO STDOUT WITH CSV HEADER;"
    )
    count = int(pd.read_csv(io.StringIO(restored_count)).iloc[0, 0])
    require(count == len(source), "Restored row count differs from source.")

    # Apply the project's existing SQL definition, not a reconstructed guess.
    sql(paths["monthly_view_sql"].read_text(encoding="utf-8"))

    result = sql(
        "COPY (SELECT * FROM analytics.v_monthly_operations) "
        "TO STDOUT WITH CSV HEADER;"
    )
    reporting = pd.read_csv(io.StringIO(result))
    reporting["period"] = pd.to_datetime(
        reporting["period"], errors="raise"
    )

    report_keys = ["carrier", "period"]
    require(not reporting[report_keys].isna().any().any(),
            "Missing SQL reporting keys.")
    require(not reporting.duplicated(report_keys).any(),
            "Duplicate SQL reporting keys.")

    actual = reporting.set_index(report_keys).sort_index()
    require(actual.index.equals(reference.index),
            "SQL and Python monthly keys differ.")

    require(set(reference.columns).issubset(actual.columns),
            "SQL reporting columns are missing.")

    checks = []
    for column in reference.columns:
        expected_values = reference[column].to_numpy(dtype=float)
        actual_values = pd.to_numeric(
            actual[column], errors="raise"
        ).to_numpy(dtype=float)

        ratio = (
            column.endswith("_pct")
            or column in {
                "late_minutes_per_100_flights",
                "avg_delay_minutes_per_delayed_arrival",
            }
        )
        matches = (
            np.isfinite(expected_values)
            & np.isfinite(actual_values)
            & np.isclose(
                expected_values, actual_values,
                rtol=0, atol=0.000001 if ratio else 0
            )
        )
        checks.append({
            "metric": column,
            "months_checked": len(matches),
            "mismatched_months": int((~matches).sum()),
            "status": "PASS" if matches.all() else "FAIL",
        })

    validation = pd.DataFrame(checks)
    validation.to_csv(OUTPUT / "sql_python_validation.csv", index=False)

    require(validation["status"].eq("PASS").all(),
            "SQL-to-Python validation failed; see saved report.")

    reporting.sort_values(report_keys).to_csv(
        OUTPUT / "monthly_operations.csv", index=False
    )
    complete("Database recovery and monthly SQL validation")

    # ---------- 5. Forecast and evaluate ----------
    log("5/6 Running the historical forecasting benchmark")

    monthly = reference.reset_index().sort_values("period")
    train = monthly[monthly["period"].dt.year.eq(2023)].copy()
    test = monthly[monthly["period"].dt.year.eq(2024)].copy()

    require(len(train) == 12 and len(test) == 12,
            "Forecast requires 12 training and 12 test months.")

    # Construct forecasts without using test-year flight counts.
    target_periods = pd.date_range("2024-01-01", "2024-12-01", freq="MS")
    prior = train["reported_flights"].to_numpy(dtype=float)

    predictions = pd.DataFrame({
        "period": target_periods,
        "forecast_origin": "2023-12-31",
        "seasonal_naive": prior,
        "calendar_adjusted_seasonal": np.rint(
            prior
            / train["period"].dt.days_in_month.to_numpy()
            * target_periods.days_in_month.to_numpy()
        ),
    })
    predictions.to_csv(OUTPUT / "frozen_forecasts.csv", index=False)

    observed = test["reported_flights"].to_numpy(dtype=float)
    require((observed > 0).all(), "Forecast evaluation denominator is zero.")

    scores, errors = [], []
    for model in ["seasonal_naive", "calendar_adjusted_seasonal"]:
        predicted = predictions[model].to_numpy(dtype=float)
        error = predicted - observed

        scores.append({
            "model": model,
            "months_tested": 12,
            "MAE_flights": np.abs(error).mean(),
            "RMSE_flights": np.sqrt(np.mean(error ** 2)),
            "MAPE_pct": np.mean(np.abs(error) / observed) * 100,
            "WAPE_pct": np.abs(error).sum() / observed.sum() * 100,
            "bias_pct": error.sum() / observed.sum() * 100,
        })

        errors.append(pd.DataFrame({
            "period": target_periods,
            "model": model,
            "actual_flights": observed,
            "forecast_flights": predicted,
            "forecast_minus_actual": error,
        }))

    accuracy = pd.DataFrame(scores)
    accuracy.to_csv(OUTPUT / "forecast_accuracy.csv", index=False)
    pd.concat(errors, ignore_index=True).to_csv(
        OUTPUT / "forecast_monthly_errors.csv", index=False
    )
    complete("Forecast backtest")

    # ---------- 6. Record successful completion ----------
    log("6/6 Saving completion evidence")
    MANIFEST["status"] = "PASS"
    MANIFEST["source_rows"] = len(source)
    MANIFEST["monthly_comparisons"] = int(
        validation["months_checked"].sum()
    )
    MANIFEST["limitations"] = [
        "Starts from preserved checkpoints; acquisition is not automated.",
        "Restores a backup rather than testing a fresh CSV-to-SQL load.",
        "Database validation compares rows and monthly aggregates, "
        "not every field or restored permissions.",
        "Desktop Power BI and Excel refresh are outside this pipeline.",
        "Forecasts are retrospective benchmarks, not official budgets.",
        "Dependency versions recorded but not pinned.",
    ]
    save_manifest()

    print("\nPIPELINE PASSED")
    display(validation)
    display(accuracy.round(2))
    print("Run folder:", OUTPUT)
    print("Isolated database retained:", TEST_DB)
    print(
        "\nTo test independence from old variables: restart the Colab "
        "session, then run only this entire cell again."
    )

except Exception as exc:
    MANIFEST["status"] = "FAIL"
    MANIFEST["error"] = str(exc)
    save_manifest()
    log("PIPELINE FAILED: " + str(exc))
    raise